# F02 필요 지식 ①~⑧ 검수 — v2.0 (패키지 구조)

입력: 원고 1개 + 선택적인 개발계획서 1개 (`.hwp` / `.hwpx`, 순서 무관, 내용으로 역할 판별). 개발계획서는 ⑤에만 필요합니다.
출력: 항목별 JSON, 요약, 지적사항, 전체 결과, 진단 정보, ZIP.

v1.0(단일 셀 노트북)의 검사 논리를 그대로 두고 `ncs_core` + 패키지 구조로 옮긴 판입니다.
차례·본문 파서는 차례 검수(`ncs_toc`), 내용체계 표·계획서 블록은 개요 검수(`ncs_overview`)의 것을 그대로 씁니다.

### 판정
- ①②③ 내비게이터 학습명: 코드 번호 없음 / '~하기' 종결 / 차례 면·내용체계 표와 일치(+묶음 간 일치, 기호 글꼴). 학습당 1행, 묶음 전부 확인.
- ④ 본문 N-M. 학습 내용명 = 차례 = 내용체계, N = 놓인 학습, M 연번. 학습당 1행.
- ⑤ 학습 목표 = 개발계획서 수행 준거. 학습 내용당 1행. 같음 O / 공백·부호만 다름 X(`whitespace_is_difference=False`면 O+참고) / 유사도 ≥ 0.8 윤문 △(직능연 협의 확인) / 그 밖·누락·본문에만 있음 X. 계획서 없으면 `-`.
- ⑥ 번호 체계 󰊱 〉 1. 〉 (1) 〉 (가) 〉 1) 〉 가) 〉 ① 〉 ㉠ 〉 ⓐ: 첫 수준·건너뜀·연번. 학습 내용당 1행. 머리말(필요 지식·수행 내용·수행 순서·재료·안전…)마다 새로 시작. 번호 문단 0개면 △(자동 번호 가능성). hwpx는 표 안 문단 제외.
- ⑦ 그림·표 번호 연속(figtab v1.4 논리): 묶음·캡션당 1행. 앞 번호 열, 뒷 번호 연속, 괄호 모양, 본문 참조, 놓인 학습과 앞 번호.
- ⑧ 출처 표기: 그림은 캡션 위 출처 줄 필수, 표는 선택(`-`). F14 템플릿 골격 5종. AI 활용 표기·구 기관명은 △.

### 처리 흐름
파일 읽기 → 차례·내용체계·본문·학습 내용 구간·계획서 목표·캡션 추출 → ①~⑧ 판정 → JSON/ZIP 저장.
기본 양식은 section0=차례·개요, section1=본문입니다. 아래 설정에서 변경할 수 있습니다.

위에서 아래로 실행하세요. Python 3.10 이상이며 추가 설치 패키지는 없습니다.
이 노트북은 `tools/build_notebook.py`로 Python 패키지에서 생성한 배포본입니다.
팀의 수정 원본은 `src/ncs_core/`(공통), `src/ncs_toc/`, `src/ncs_overview/`, `src/ncs_knowledge/`이고, 수정 후 노트북을 다시 생성합니다.

- `O`: 충족, `X`: 위반, `△`: 사람 확인, `-`: 해당 없음 또는 대조 생략.
- 행 판정(검수결과): X가 하나라도 있으면 `부적합`, X 없이 △가 있으면 `판단불가`, 나머지는 `적합`.
- `*_00_요약.json`의 `검수결과`는 검수 확인서 표현대로 적합/부적합/판단불가로 종합한다(부적합 > 판단불가 > 적합).


## 1. 실행 환경 준비
임시 폴더에 이 배포본의 패키지를 구성합니다. 기존 프로젝트 파일은 덮어쓰지 않습니다.


In [ ]:
from pathlib import Path
import importlib
import json
import sys
import tempfile

RUNTIME_DIR = Path(tempfile.mkdtemp(prefix="ncs_knowledge_"))
# 실패하거나 재실행해도 이전 결과를 다시 쓰지 않도록 초기화합니다.
result = saved = None
PATHS = []


def write_module(relative_path, source):
    path = RUNTIME_DIR / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source, encoding="utf-8")


print("검수 엔진 위치:", RUNTIME_DIR)


## 2. 검수 엔진
각 셀은 한 모듈의 소스입니다(접혀 있음). 사용만 할 때는 그대로 실행하세요.


In [ ]:
# 모듈: ncs_core/__init__.py
write_module('ncs_core/__init__.py', r'''"""F02 검수 공통 코어 (Python 3.10+, 추가 설치 패키지 없음).

두 검수 패키지(ncs_toc, ncs_position)가 공유하는 부분:
문서 읽기, 문자 정규화·판정, 입력 역할 판별, 모듈 코드, 규칙 실행, 결과 저장.
검수 정책(무엇을 X로 볼지)은 여기에 두지 않는다.
"""

from .documents import Document, Table, load_document
from .inputs import detect_role, select_documents, validate_distinct
from .models import ReviewResult, SavedReport
from .module_code import resolve_module_code, validate_module_code
from .reporting import save_report
from .rules import Rule, run_rules
from .text import FAIL, NA, O, PASS, PUA, TRI, UNDECIDED, VERDICTS, X, flatten, make_normalizer, verdict, visible

__all__ = [
    "Document",
    "Table",
    "load_document",
    "detect_role",
    "select_documents",
    "validate_distinct",
    "ReviewResult",
    "SavedReport",
    "resolve_module_code",
    "validate_module_code",
    "save_report",
    "Rule",
    "run_rules",
    "NA",
    "PASS",
    "FAIL",
    "UNDECIDED",
    "VERDICTS",
    "O",
    "PUA",
    "TRI",
    "X",
    "flatten",
    "make_normalizer",
    "verdict",
    "visible",
]
__version__ = "1.0.0"
''')


In [ ]:
# 모듈: ncs_core/documents.py
write_module('ncs_core/documents.py', r'''"""문서를 한 번 읽어 Document로 만든다. 검수 정책이나 Colab에 의존하지 않는다."""

import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from . import hwp, hwpx

Table = dict[str, Any]


@dataclass(frozen=True)
class Document:
    """읽은 문서. 모든 텍스트는 연속 공백이 하나로 정리된 상태다.

    sections: 섹션별 문단 텍스트(표 셀 안 문단 포함, 문서 순서).
    tables:   표 목록(문서 순서). 'styles'는 hwpx일 때만 채워진다.
    images:   (섹션, 문단 인덱스, 원본 그림 이름). hwpx만.
    """

    path: Path
    format: str
    sections: list[list[str]]
    tables: list[Table] = field(default_factory=list)
    images: list[tuple[int, int, str]] = field(default_factory=list)

    @property
    def has_styles(self) -> bool:
        return self.format == "hwpx"

    def section_tables(self, index: int) -> list[Table]:
        return [t for t in self.tables if t["section"] == index]


def detect_format(path) -> str:
    """파일 내용으로 hwp/hwpx를 판별한다. 둘 다 아니면 ValueError."""
    path = Path(path)
    with path.open("rb") as stream:
        head = stream.read(8)
    if head == hwp.OLE_MAGIC:
        if not hwp.is_hwp(path.read_bytes()):
            raise ValueError(f"OLE 파일이지만 HWP 서명이 아님: {path}")
        return "hwp"
    if head[:2] == b"PK":
        if hwpx.is_hwpx(path):
            return "hwpx"
        raise ValueError(f"zip이지만 hwpx가 아님: {path}")
    raise ValueError(f"hwp/hwpx 둘 다 아님 (머리 바이트 {head!r}): {path}")


def _clean(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def load_document(path) -> Document:
    path = Path(path)
    fmt = detect_format(path)
    if fmt == "hwp":
        sections, tables = hwp.load_hwp(path.read_bytes())
        images = []
    else:
        sections, tables, images = hwpx.load_hwpx(path)
    sections = [[_clean(p) for p in section] for section in sections]
    for table in tables:
        table["cells"] = {key: _clean(value) for key, value in table["cells"].items()}
    if not any(line for section in sections for line in section):
        raise ValueError(f"문단 텍스트를 추출하지 못했습니다: {path}")
    return Document(path, fmt, sections, tables, images)
''')


In [ ]:
# 모듈: ncs_core/hwp.py
write_module('ncs_core/hwp.py', r'''"""HWP(바이너리, OLE/CFB) 읽기. 문단 텍스트와 표 셀 텍스트만 읽고 서식은 읽지 않는다."""

import struct
import zlib
from collections.abc import Iterator

OLE_MAGIC = b"\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1"
HWP_SIGNATURE = b"HWP Document File"
END_OF_CHAIN = (0xFFFFFFFE, 0xFFFFFFFF)

TAG_PARA_TEXT, TAG_LIST_HEADER, TAG_TABLE = 0x43, 0x48, 0x4D
# 문단 텍스트의 제어 문자 중 2바이트짜리. 나머지 제어 문자는 16바이트.
SHORT_CONTROL_CHARS = (0, 10, 13, 24, 25, 26, 27, 28, 29, 30, 31)


def read_cfb(data: bytes):
    """CFB 컨테이너를 열어 (read_stream, 스트림 이름 목록)을 돌려준다.

    섹터 n은 파일 오프셋 (n+1)*섹터크기에서 시작한다(CFB 규격).
    순환하는 섹터 체인은 손상으로 보고 예외를 낸다.
    """
    header = data[:512]
    sector_size = 1 << struct.unpack_from("<H", header, 0x1E)[0]
    mini_sector_size = 1 << struct.unpack_from("<H", header, 0x20)[0]
    dir_start = struct.unpack_from("<I", header, 0x30)[0]
    mini_cutoff = struct.unpack_from("<I", header, 0x38)[0]
    minifat_start = struct.unpack_from("<I", header, 0x3C)[0]
    difat_start = struct.unpack_from("<I", header, 0x44)[0]
    difat = list(struct.unpack_from("<109I", header, 0x4C))
    entries_per_sector = sector_size // 4

    def sector(n: int) -> bytes:
        return data[(n + 1) * sector_size : (n + 2) * sector_size]

    next_difat, seen = difat_start, set()
    while next_difat not in END_OF_CHAIN:
        if next_difat in seen:
            raise ValueError("손상된 HWP: DIFAT 순환 참조")
        seen.add(next_difat)
        values = struct.unpack(f"<{entries_per_sector}I", sector(next_difat))
        difat += values[:-1]
        next_difat = values[-1]

    fat: list[int] = []
    for fat_sector in difat:
        if fat_sector not in END_OF_CHAIN:
            fat += struct.unpack(f"<{entries_per_sector}I", sector(fat_sector))

    def chain(start: int, table: list[int]) -> list[int]:
        out, current, visited = [], start, set()
        while current not in END_OF_CHAIN:
            if current in visited or current >= len(table):
                raise ValueError("손상된 HWP: 잘못된 섹터 체인")
            visited.add(current)
            out.append(current)
            current = table[current]
        return out

    def read_chain(start: int, size: int) -> bytes:
        return b"".join(sector(s) for s in chain(start, fat))[:size]

    directory = b"".join(sector(s) for s in chain(dir_start, fat))
    entries = []
    for i in range(len(directory) // 128):
        entry = directory[i * 128 : (i + 1) * 128]
        name_length = struct.unpack_from("<H", entry, 0x40)[0]
        entries.append(
            (
                entry[: max(name_length - 2, 0)].decode("utf-16le", "ignore"),
                entry[0x42],
                struct.unpack_from("<I", entry, 0x74)[0],
                struct.unpack_from("<Q", entry, 0x78)[0],
            )
        )
    ministream = read_chain(entries[0][2], entries[0][3])
    minifat: list[int] = []
    for s in chain(minifat_start, fat):
        minifat += struct.unpack(f"<{entries_per_sector}I", sector(s))

    def read_stream(name: str) -> bytes | None:
        for entry_name, kind, start, size in entries:
            if entry_name == name and kind == 2:
                if size < mini_cutoff:
                    return b"".join(
                        ministream[s * mini_sector_size : (s + 1) * mini_sector_size]
                        for s in chain(start, minifat)
                    )[:size]
                return read_chain(start, size)
        return None

    return read_stream, [name for name, kind, *_ in entries if kind == 2]


def records(buffer: bytes) -> Iterator[tuple[int, int, bytes]]:
    """본문 스트림을 (tag, level, payload)로 나눈다. 잘린 레코드는 예외."""
    i = 0
    while i + 4 <= len(buffer):
        head = struct.unpack_from("<I", buffer, i)[0]
        i += 4
        tag, level, size = head & 0x3FF, (head >> 10) & 0x3FF, head >> 20
        if size == 0xFFF:
            if i + 4 > len(buffer):
                raise ValueError("손상된 HWP: 확장 레코드 길이 누락")
            size = struct.unpack_from("<I", buffer, i)[0]
            i += 4
        if i + size > len(buffer):
            raise ValueError("손상된 HWP: 레코드 잘림")
        yield tag, level, buffer[i : i + size]
        i += size


def para_text(payload: bytes) -> str:
    """PARA_TEXT 레코드에서 문자열을 꺼낸다. 탭·줄바꿈은 공백으로."""
    out, i, n = [], 0, len(payload)
    while i + 2 <= n:
        code = struct.unpack_from("<H", payload, i)[0]
        if code < 32:
            i += 2 if code in SHORT_CONTROL_CHARS else 16
            if code in (9, 10, 13):
                out.append(" ")
            continue
        out.append(chr(code))
        i += 2
    return "".join(out)


def is_hwp(data: bytes) -> bool:
    """OLE 파일이고 FileHeader 스트림의 서명이 HWP인가."""
    if data[:8] != OLE_MAGIC:
        return False
    try:
        read_stream, _ = read_cfb(data)
        header = read_stream("FileHeader")
    except (ValueError, struct.error, IndexError):
        return False
    return header is not None and header[:32].rstrip(b"\x00") == HWP_SIGNATURE


def load_hwp(data: bytes) -> tuple[list[list[str]], list[dict]]:
    """(섹션별 문단 목록, 표 목록)을 돌려준다.

    문단 목록은 모든 PARA_TEXT 레코드(표 셀 안 포함)를 순서대로 담는다.
    표의 `paragraph`는 그 표를 가진 문단의 인덱스(같은 기준)다.
    """
    read_stream, names = read_cfb(data)
    file_header = read_stream("FileHeader")
    if file_header is None:
        raise ValueError("HWP FileHeader 스트림이 없음")
    flags = struct.unpack_from("<I", file_header, 36)[0]
    if flags & 2:
        raise ValueError("암호화된 hwp는 읽을 수 없음")
    section_names = sorted(
        (n for n in names if n.startswith("Section")), key=lambda s: int(s[7:])
    )
    sections, tables = [], []
    for section_index, name in enumerate(section_names):
        raw = read_stream(name)
        if flags & 1:
            raw = zlib.decompress(raw, -15)
        paragraphs: list[str] = []
        stack: list[tuple[int, dict]] = []  # (level, table)
        cell: tuple[int, dict, tuple[int, int]] | None = None
        for tag, level, payload in records(raw):
            if tag == TAG_PARA_TEXT:
                paragraphs.append(para_text(payload))
                if cell and level == cell[0] + 1:
                    table, key = cell[1], cell[2]
                    table["cells"][key] = (table["cells"][key] + " " + paragraphs[-1]).strip()
            elif tag == TAG_TABLE:
                rows, cols = struct.unpack_from("<HH", payload, 4)
                table = {
                    "section": section_index,
                    "paragraph": max(len(paragraphs) - 1, 0),
                    "rows": rows,
                    "cols": cols,
                    "cells": {},
                    "styles": None,
                }
                while stack and stack[-1][0] >= level:
                    stack.pop()
                stack.append((level, table))
                tables.append(table)
            elif tag == TAG_LIST_HEADER:
                while stack and stack[-1][0] > level:
                    stack.pop()  # 안쪽 표가 끝남
                if stack and stack[-1][0] == level and len(payload) >= 16:
                    col, row = struct.unpack_from("<HH", payload, 8)
                    table = stack[-1][1]
                    table["cells"].setdefault((row, col), "")
                    cell = (level, table, (row, col))
                else:
                    cell = None
        sections.append(paragraphs)
    return sections, tables
''')


In [ ]:
# 모듈: ncs_core/hwpx.py
write_module('ncs_core/hwpx.py', r'''"""HWPX(zip+XML) 읽기. 표준 ElementTree만 사용한다.

한 섹션 XML을 한 번 순회해 문단 텍스트, 표(셀 텍스트와 서식), 그림 이름을 함께 꺼낸다.
"""

import re
import zipfile
from xml.etree import ElementTree as ET

HP = "{http://www.hancom.co.kr/hwpml/2011/paragraph}"
HH = "{http://www.hancom.co.kr/hwpml/2011/head}"
HC = "{http://www.hancom.co.kr/hwpml/2011/core}"

SECTION_NAME = re.compile(r"Contents/section(\d+)\.xml")
IMAGE_NAME = re.compile(r"원본 그림의 이름:\s*([^\n]+)")
NO_FILL = (None, "", "none", "#FFFFFF", "#ffffff")


def is_hwpx(path) -> bool:
    with zipfile.ZipFile(path) as archive:
        return any(SECTION_NAME.fullmatch(n) for n in archive.namelist())


def section_names(archive: zipfile.ZipFile) -> list[str]:
    names = [n for n in archive.namelist() if SECTION_NAME.fullmatch(n)]
    return sorted(names, key=lambda n: int(SECTION_NAME.fullmatch(n).group(1)))


def para_text(paragraph: ET.Element) -> str:
    """변경추적 반영본 텍스트. deleteBegin~deleteEnd 구간은 버린다.

    표 셀·캡션 안에 중첩된 <hp:p>는 내려가지 않는다. 그 문단들은 섹션 순회에서
    따로 나오므로, 내려가면 바깥 문단에 셀 텍스트가 전부 합쳐진다.
    <hp:t> 안 인라인 요소 뒤의 tail 텍스트는 포함한다.
    """
    out, state = [], None

    def walk(element: ET.Element) -> None:
        nonlocal state
        for child in element:
            tag = child.tag.replace(HP, "")
            if tag == "p":
                continue
            if tag == "deleteBegin":
                state = "del"
            elif tag in ("deleteEnd", "insertEnd"):
                state = None
            elif tag == "insertBegin":
                state = "ins"
            if state != "del" and tag == "t":
                out.append(child.text or "")
            walk(child)
            if state != "del" and child.tail and element.tag == HP + "t":
                out.append(child.tail)

    walk(paragraph)
    return "".join(out)


def header_styles(archive: zipfile.ZipFile) -> tuple[dict, dict]:
    """header.xml → (charPr id→{bold, font}, borderFill id→면색 또는 None)."""
    root = ET.fromstring(archive.read("Contents/header.xml"))
    fonts = {}
    for face in root.iter(HH + "fontface"):
        if face.get("lang") == "HANGUL":
            for font in face.findall(HH + "font"):
                fonts[font.get("id")] = font.get("face")
    char_props = {}
    for prop in root.iter(HH + "charPr"):
        ref = prop.find(HH + "fontRef")
        char_props[prop.get("id")] = {
            "bold": prop.find(HH + "bold") is not None,
            "font": fonts.get(ref.get("hangul")) if ref is not None else None,
        }
    fills = {}
    for border_fill in root.iter(HH + "borderFill"):
        brush = border_fill.find(".//" + HC + "winBrush")
        color = brush.get("faceColor") if brush is not None else None
        fills[border_fill.get("id")] = None if color in NO_FILL else color.upper()
    return char_props, fills


def _cell_runs(cell: ET.Element, own_paragraphs: list[ET.Element], char_props: dict) -> list[dict]:
    runs = []
    for paragraph in own_paragraphs:
        for run in paragraph.iter(HP + "run"):
            text = re.sub(r"\s+", " ", "".join(t.text or "" for t in run.iter(HP + "t"))).strip()
            if text:
                runs.append(
                    {
                        "text": text,
                        **char_props.get(run.get("charPrIDRef"), {"bold": False, "font": None}),
                    }
                )
    return runs


def parse_section(root: ET.Element, section_index: int, styles: tuple[dict, dict] | None):
    """(문단 텍스트 목록, 표 목록, 그림 목록)을 돌려준다.

    표: {'section', 'paragraph', 'rows', 'cols', 'cells': {(r,c): 텍스트},
         'styles': {(r,c): {'fill', 'runs': [{'text','font','bold'}]}} (styles 없으면 None)}
    그림: (문단 인덱스, 원본 그림 이름)
    """
    parents = {child: parent for parent in root.iter() for child in parent}
    paragraphs = list(root.iter(HP + "p"))
    positions = {p: i for i, p in enumerate(paragraphs)}

    def owner_paragraph(element: ET.Element) -> ET.Element | None:
        current = parents.get(element)
        while current is not None and current.tag != HP + "p":
            current = parents.get(current)
        return current

    def owner_cell(paragraph: ET.Element) -> ET.Element | None:
        current = parents.get(paragraph)
        while current is not None and current.tag != HP + "tc":
            current = parents.get(current)
        return current

    images = []
    for comment in root.iter(HP + "shapeComment"):
        match = IMAGE_NAME.search(comment.text or "")
        paragraph = owner_paragraph(comment)
        if match and paragraph in positions:
            images.append((positions[paragraph], match.group(1).strip()))

    tables = []
    for table in root.iter(HP + "tbl"):
        paragraph = owner_paragraph(table)
        if paragraph not in positions:
            continue
        cells, cell_styles = {}, {}
        for row in table.findall(HP + "tr"):
            for cell in row.findall(HP + "tc"):
                address = cell.find(HP + "cellAddr")
                if address is None:
                    raise ValueError("HWPX 표 셀에 cellAddr가 없습니다")
                key = (int(address.get("rowAddr")), int(address.get("colAddr")))
                # 중첩 표는 자체 표로 처리하므로 바깥 셀에 합치지 않는다.
                own = [p for p in cell.iter(HP + "p") if owner_cell(p) is cell]
                cells[key] = re.sub(r"\s+", " ", " ".join(para_text(p) for p in own)).strip()
                if styles is not None:
                    char_props, fills = styles
                    cell_styles[key] = {
                        "fill": fills.get(cell.get("borderFillIDRef")),
                        "runs": _cell_runs(cell, own, char_props),
                    }
        tables.append(
            {
                "section": section_index,
                "paragraph": positions[paragraph],
                "rows": int(table.get("rowCnt") or 0),
                "cols": int(table.get("colCnt") or 0),
                "cells": cells,
                "styles": cell_styles if styles is not None else None,
            }
        )
    return [para_text(p) for p in paragraphs], tables, images


def load_hwpx(path):
    """(섹션별 문단, 표, 그림) — 그림은 (섹션, 문단, 이름)."""
    sections, tables, images = [], [], []
    with zipfile.ZipFile(path) as archive:
        try:
            styles = header_styles(archive)
        except KeyError:
            styles = None
        for index, name in enumerate(section_names(archive)):
            root = ET.fromstring(archive.read(name))
            paragraphs, section_tables, section_images = parse_section(root, index, styles)
            sections.append(paragraphs)
            tables.extend(section_tables)
            images.extend((index, paragraph, image) for paragraph, image in section_images)
    return sections, tables, images
''')


In [ ]:
# 모듈: ncs_core/inputs.py
write_module('ncs_core/inputs.py', r'''"""입력 파일의 역할 판별과 검증. Colab UI와 독립적이다."""

import hashlib
from pathlib import Path

from .documents import Document, load_document

ROLE_LABELS = {"manuscript": "원고", "plan": "개발계획서"}
MANUSCRIPT_MARKERS = ("차 례", "차례", "학습모듈의 개요", "NCS-학습모듈의 위치")


def detect_role(sections: list[list[str]], head_lines: int = 60) -> str:
    """첫 섹션 머리 문단으로 역할을 판별한다: 'plan' / 'manuscript' / 'unknown'."""
    head = " ".join(sections[0][:head_lines]) if sections else ""
    if "개발계획서" in head:
        return "plan"
    if any(marker in head for marker in MANUSCRIPT_MARKERS):
        return "manuscript"
    return "unknown"


def file_digest(path) -> bytes:
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.digest()


def validate_distinct(paths: dict[str, str | Path]) -> None:
    """역할별 경로가 존재하고 비어 있지 않으며 서로 다른 내용인지 확인한다."""
    for role, path in paths.items():
        label = ROLE_LABELS.get(role, role)
        if not path:
            raise ValueError(f"{label} 파일 경로를 지정하세요")
        path = Path(path)
        if not path.is_file():
            raise ValueError(f"{label} 파일을 찾을 수 없습니다: {path}")
        if path.stat().st_size == 0:
            raise ValueError(f"{label} 파일이 비어 있습니다: {path}")
    digests = {role: file_digest(path) for role, path in paths.items()}
    if len(set(digests.values())) != len(digests):
        raise ValueError("입력 파일들의 내용이 동일합니다. 실제 원고와 개발계획서를 각각 선택하세요")


def select_documents(
    paths,
    roles: dict | None = None,
    *,
    required: tuple[str, ...] = ("manuscript",),
) -> dict[str, Document]:
    """경로 목록을 읽어 {'manuscript': Document, 'plan': Document}로 나눈다.

    roles={경로: 'plan'|'manuscript'}로 자동 판별을 덮어쓴다.
    required에 있는 역할이 없거나, 한 역할에 파일이 둘 이상이면 ValueError.
    """
    overrides = {str(Path(p).resolve()): r for p, r in (roles or {}).items()}
    if set(overrides.values()) - set(ROLE_LABELS):
        raise ValueError("roles 값은 'manuscript' 또는 'plan'이어야 합니다")
    documents: dict[str, Document] = {}
    seen: set[str] = set()
    for path in paths:
        key = str(Path(path).resolve())
        if key in seen:
            raise ValueError(f"같은 입력 파일이 중복 지정되었습니다: {path}")
        seen.add(key)
        document = load_document(path)
        role = overrides.get(key, detect_role(document.sections))
        if role not in ROLE_LABELS:
            raise ValueError(f"파일 역할을 판단하지 못했습니다: {path}. roles로 역할을 지정하세요")
        if role in documents:
            raise ValueError(f"{ROLE_LABELS[role]} 파일이 둘 이상입니다. 역할마다 한 개씩 선택하세요")
        documents[role] = document
    unused = overrides.keys() - seen
    if unused:
        raise ValueError(f"roles에 입력 목록에 없는 경로가 있습니다: {sorted(unused)}")
    for role in required:
        if role not in documents:
            raise ValueError(f"{ROLE_LABELS[role]} 파일이 필요합니다")
    if len(documents) > 1:
        validate_distinct({role: doc.path for role, doc in documents.items()})
    return documents
''')


In [ ]:
# 모듈: ncs_core/models.py
write_module('ncs_core/models.py', r'''"""파이프라인 경계의 공통 데이터 계약. 체크리스트 행의 한국어 JSON 키는 기존과 같다."""

from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Literal, TypedDict

ChecklistItem = TypedDict(
    "ChecklistItem",
    {
        "item_id": str,
        "region_path": str,
        "원고": Any,
        "체크": dict[str, Literal["O", "X", "△", "-"]],
        "판정": str,
        "비고": dict[str, str] | list[str],
        "수정 제안": Any,
    },
)


@dataclass
class ReviewResult:
    """한 번의 검사 결과. 저장 형식과 무관하다."""

    module_code: str
    checklist: list[ChecklistItem]
    findings: list[dict]
    diagnostics: dict[str, Any] = field(default_factory=dict)
    context: Any = None


@dataclass(frozen=True)
class SavedReport:
    directory: Path
    zip_path: Path
    summary: dict[str, Any]
''')


In [ ]:
# 모듈: ncs_core/module_code.py
write_module('ncs_core/module_code.py', r'''"""모듈 코드 검증과 파일명 추출. input()을 호출하지 않는다."""

import re
from pathlib import Path

CODE_PATTERN = r"LM[0-9]{10}_[0-9]+v[0-9]+"
CODE_IN_NAME = re.compile(r"(?<![A-Za-z0-9])" + CODE_PATTERN + r"(?![A-Za-z0-9])")


def validate_module_code(code: str) -> str:
    code = (code or "").strip()
    if not re.fullmatch(CODE_PATTERN, code):
        raise ValueError("모듈 코드는 LM + 숫자 10자리 + _ + 숫자 + v + 숫자 형식이어야 합니다")
    if int(code[8:10]) < 1 or int(code[10:12]) < 1:
        raise ValueError("세분류 번호와 능력단위 번호는 01 이상이어야 합니다")
    return code


def resolve_module_code(manuscript_path, override: str | None = "") -> str:
    """override가 있으면 검증해 쓰고, 없으면 원고 파일명에서 하나뿐인 코드를 찾는다."""
    if override and override.strip():
        return validate_module_code(override)
    candidates = list(dict.fromkeys(CODE_IN_NAME.findall(Path(manuscript_path).name)))
    if len(candidates) != 1:
        raise ValueError("원고 파일명의 모듈 코드가 없거나 여러 개입니다. module_code를 지정하세요")
    return validate_module_code(candidates[0])
''')


In [ ]:
# 모듈: ncs_core/reporting.py
write_module('ncs_core/reporting.py', r'''"""검사 결과를 항목별 JSON과 ZIP으로 저장한다. 파일 이름 규칙은 두 패키지가 같다.

<out_dir>/<코드>_<실행별 폴더>/
    <코드>_<항목 꼬리말>.json   항목별 체크리스트
    <코드>_<이름>_00_요약.json  검수결과(적합/부적합/판단불가) 종합과 항목별 집계
    <코드>_<이름>_지적사항.json  X·△만
    <코드>_<이름>_전체.json     모든 체크리스트 행
    <코드>_<이름>_진단.json     입력·읽기 상태
    <코드>_<이름>_검수결과.zip
"""

import json
import tempfile
import zipfile
from collections.abc import Sequence
from pathlib import Path

from .models import ReviewResult, SavedReport
from .module_code import validate_module_code
from .text import FAIL, PASS, UNDECIDED, VERDICTS


def overall_verdict(verdicts) -> str:
    """행 판정들의 종합: 부적합이 하나라도 있으면 부적합, 없고 판단불가가 있으면 판단불가, 나머지는 적합."""
    verdicts = list(verdicts)
    if FAIL in verdicts:
        return FAIL
    if UNDECIDED in verdicts:
        return UNDECIDED
    return PASS


def save_report(
    result: ReviewResult,
    out_dir,
    *,
    name: str,
    item_files: Sequence[tuple[str, str]],
    other_tail: str,
    per_run_dir: bool = True,
) -> SavedReport:
    """name: 결과 이름('차례'/'위치'). item_files: (item_id 접두, 파일 꼬리말).

    per_run_dir=True이면 실행마다 새 하위 폴더를 만들어 이전 결과를 덮어쓰지 않는다.
    False이면 out_dir에 바로 쓰고 같은 코드의 이전 결과를 갱신한다.
    item_files에 없는 item_id는 other_tail 파일로 모인다.
    """
    code = validate_module_code(result.module_code)
    root = Path(out_dir).resolve()
    root.mkdir(parents=True, exist_ok=True)
    directory = Path(tempfile.mkdtemp(prefix=f"{code}_", dir=root)) if per_run_dir else root

    buckets: dict[str, list] = {tail: [] for _, tail in item_files}
    by_length = sorted(item_files, key=lambda pair: -len(pair[0]))
    for row in result.checklist:
        tail = next((t for prefix, t in by_length if row["item_id"].startswith(prefix)), other_tail)
        buckets.setdefault(tail, []).append(row)

    paths: list[Path] = []
    summary: dict = {}

    def write(tail: str, value) -> Path:
        path = directory / f"{code}_{tail}.json"
        path.write_text(json.dumps(value, ensure_ascii=False, indent=1), encoding="utf-8")
        paths.append(path)
        return path

    for tail, rows in buckets.items():
        path = write(tail, rows)
        verdicts = [row["판정"] for row in rows]
        summary[tail] = {"검수결과": overall_verdict(verdicts), "행 수": {v: verdicts.count(v) for v in VERDICTS}, "파일": path.name}
    summary = {"검수결과": overall_verdict(row["판정"] for row in result.checklist), "항목": summary}
    write(f"{name}_00_요약", summary)
    write(f"{name}_지적사항", result.findings)
    write(f"{name}_전체", result.checklist)
    write(f"{name}_진단", result.diagnostics)

    zip_path = directory / f"{code}_{name}_검수결과.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in paths:
            archive.write(path, path.name)
    return SavedReport(directory, zip_path, summary)
''')


In [ ]:
# 모듈: ncs_core/rules.py
write_module('ncs_core/rules.py', r'''"""규칙 등록과 실행. 규칙 함수는 컨텍스트 하나를 받아 체크리스트 행 목록을 돌려준다."""

from collections.abc import Callable, Sequence
from dataclasses import dataclass
from typing import Any


@dataclass(frozen=True)
class Rule:
    """key: 규칙 식별자(중복 불가). item_prefix: 반환 행의 item_id가 시작해야 하는 문자열."""

    key: str
    item_prefix: str
    run: Callable[[Any], list[dict]]


def run_rules(context: Any, rules: Sequence[Rule]) -> list[dict]:
    keys = [rule.key for rule in rules]
    if len(keys) != len(set(keys)):
        raise ValueError("중복된 규칙 key가 있습니다")
    rows: list[dict] = []
    for rule in rules:
        result = rule.run(context)
        for row in result:
            if not row["item_id"].startswith(rule.item_prefix):
                raise ValueError(f"{rule.key} 규칙이 다른 item_id를 반환했습니다: {row['item_id']}")
        rows.extend(result)
    return rows
''')


In [ ]:
# 모듈: ncs_core/tables.py
write_module('ncs_core/tables.py', r'''"""Document.tables(문서 순서의 표 목록)를 다루는 작은 도우미."""

from .documents import Table


def column(table: Table, col: int) -> list[str]:
    """해당 열의 비어 있지 않은 셀 텍스트를 행 순서대로."""
    return [v for (r, c), v in sorted(table["cells"].items()) if c == col and v]


def column_cells(table: Table, col: int, header: str | None = None) -> list[tuple[int, str]]:
    """해당 열의 비어 있지 않은 셀을 (행, 텍스트)로, header 텍스트는 제외."""
    return [
        (r, v)
        for (r, c), v in sorted(table["cells"].items())
        if c == col and v and v != header
    ]


def header_columns(table: Table, names: tuple[str, ...]) -> dict[str, int]:
    """names에 있는 머리글 셀 텍스트 → 열 번호 (먼저 나온 것 우선)."""
    found: dict[str, int] = {}
    for (r, c), value in sorted(table["cells"].items()):
        if value in names and value not in found:
            found[value] = c
    return found


def find_table(tables: list[Table], must_contain: str, after: Table | None = None) -> Table | None:
    """must_contain을 포함한 셀이 있는 첫 표 (after 표 다음부터)."""
    start = tables.index(after) + 1 if after is not None else 0
    for table in tables[start:]:
        if any(must_contain in v for v in table["cells"].values()):
            return table
    return None
''')


In [ ]:
# 모듈: ncs_core/text.py
write_module('ncs_core/text.py', r'''"""문자 정규화·비교·판정·지적사항 변환. 기존 결과와 호환된다.

정규화에서 어떤 머리 번호를 지우는지는 패키지마다 다르므로
`make_normalizer(prefix)`로 각 패키지가 자기 규칙의 `norm`/`same`을 만든다.
"""

import re
from collections.abc import Callable, Iterable

PUA = re.compile(r"[-]")
O, X, TRI, NA = "O", "X", "△", "-"
# 검수 확인서의 검수결과 표현. 한 항목의 판정은 이 셋 중 하나다.
PASS, FAIL, UNDECIDED = "적합", "부적합", "판단불가"
VERDICTS = (PASS, FAIL, UNDECIDED)

Normalizer = Callable[[str | None], str]


def make_normalizer(prefix: str | None = None) -> tuple[Normalizer, Callable[..., bool]]:
    """(norm, same)을 만든다.

    norm: 기호 글꼴(PUA) 문자를 가운뎃점으로 바꾸고, 양끝 공백과 머리 번호(prefix)를
    지운 뒤 연속 공백을 하나로 만든다.
    same: 두 문자열의 norm 결과가 같은가.
    """
    head = re.compile(prefix) if prefix else None

    def norm(text: str | None) -> str:
        value = PUA.sub("·", text or "").strip()
        if head is not None:
            value = head.sub("", value)
        return re.sub(r"\s+", " ", value)

    def same(expected: str | None, actual: str | None) -> bool:
        return norm(expected) == norm(actual)

    return norm, same


def visible(text: str | None) -> str:
    """PUA 문자를 ⟨U+XXXX⟩로 드러내 결과에 적는다."""
    return PUA.sub(lambda m: f"⟨U+{ord(m.group()):04X}⟩", text or "")


def verdict(checks: dict[str, str]) -> str:
    """X가 있으면 부적합, X 없이 △가 있으면 판단불가, 나머지는 적합. '-'는 무시."""
    values = [v for v in checks.values() if v != NA]
    if X in values:
        return FAIL
    if TRI in values:
        return UNDECIDED
    return PASS


def flatten(checklist: Iterable[dict]) -> list[dict]:
    """체크 항목 중 X·△만 지적사항 행으로 편다 (기존 키 유지)."""
    findings = []
    for row in checklist:
        for name, value in row["체크"].items():
            if value not in (X, TRI):
                continue
            notes = row["비고"]
            reason = notes.get(name, "") if isinstance(notes, dict) else "; ".join(notes)
            findings.append(
                {
                    "item_id": row["item_id"],
                    "check": name,
                    "region_path": row["region_path"],
                    "verdict": FAIL if value == X else UNDECIDED,
                    "evidence_quote": row["원고"],
                    "reason": reason,
                    "suggestion": row["수정 제안"],
                }
            )
    return findings
''')


In [ ]:
# 모듈: ncs_knowledge/__init__.py
write_module('ncs_knowledge/__init__.py', r'''"""F02 필요 지식 ①~⑧ 검수. 읽기·판정·저장은 ncs_core, 차례·본문 파서는 ncs_toc, 내용체계·계획서 파서는 ncs_overview를 쓴다."""

from ncs_core import ReviewResult, SavedReport

from .config import CheckConfig
from .pipeline import evaluate_review, prepare_review, run_review
from .registry import DEFAULT_RULES
from .reporting import save_results

__all__ = [
    "CheckConfig",
    "DEFAULT_RULES",
    "ReviewResult",
    "SavedReport",
    "prepare_review",
    "evaluate_review",
    "run_review",
    "save_results",
]
__version__ = "2.0.0"  # v1.0(단일 셀 노트북)을 패키지 구조로 옮긴 판
''')


In [ ]:
# 모듈: ncs_knowledge/__main__.py
write_module('ncs_knowledge/__main__.py', r'''"""python -m ncs_knowledge 원고.hwpx [개발계획서.hwp] [--module-code LM...] [--out 결과]"""

import argparse
import json

from . import CheckConfig, run_review, save_results


def main() -> None:
    parser = argparse.ArgumentParser(description="F02 필요 지식 ①~⑧ 검수")
    parser.add_argument("paths", nargs="+", help="원고와 (선택) 개발계획서. 순서 무관, 내용으로 역할 판별")
    parser.add_argument("--manuscript", help="자동 역할 판별 대신 원고로 지정할 입력 경로")
    parser.add_argument("--plan", help="자동 역할 판별 대신 개발계획서로 지정할 입력 경로")
    parser.add_argument("--module-code", default="", help="비우면 원고 파일명에서 추출")
    parser.add_argument("--front-section", type=int, default=0)
    parser.add_argument("--body-section", type=int, default=1)
    parser.add_argument("--out", default="결과")
    args = parser.parse_args()
    roles = {}
    if args.manuscript:
        roles[args.manuscript] = "manuscript"
    if args.plan:
        roles[args.plan] = "plan"
    try:
        config = CheckConfig(front_section=args.front_section, body_section=args.body_section)
        result = run_review(args.paths, roles=roles, module_code=args.module_code, config=config)
        saved = save_results(result, args.out)
    except (OSError, ValueError) as error:
        parser.exit(2, f"검수 실행 실패: {error}\n")
    print(json.dumps({"요약": saved.summary, "진단": result.diagnostics}, ensure_ascii=False, indent=2))
    print(f"지적사항 {len(result.findings)}건 · {saved.zip_path}")


if __name__ == "__main__":
    main()
''')


In [ ]:
# 모듈: ncs_knowledge/config.py
write_module('ncs_knowledge/config.py', r'''"""필요 지식 검수의 조정 가능한 양식 가정. 차례 파서(ncs_toc)의 CheckConfig를 함께 만든다."""
from dataclasses import dataclass

from ncs_toc.config import CheckConfig as TocConfig


@dataclass(frozen=True)
class CheckConfig:
    front_section: int = 0            # 차례·개요 섹션
    body_section: int = 1             # 본문 섹션
    goal_lookahead: int = 8           # ⑤ N-M. 제목 뒤 몇 줄 안에서 '학습 목표' 줄을 찾는가
    goal_similarity: float = 0.8      # ⑤ 이 이상이면 윤문 수준 차이(△), 미만이면 불일치(X)
    whitespace_is_difference: bool = True  # ⑤ 공백·문장부호만 다른 학습 목표를 X로 볼지(False면 O + 참고 비고)
    figtab_min_agreement: float = 0.5 # ⑦ 캡션 앞 번호와 학습 경계 일치율이 이 미만이면 '-위치' 검사 생략
    source_lookback: int = 3          # ⑧ 그림 캡션 위 몇 줄 안에서 출처 줄을 찾는가
    source_lookahead: int = 80        # ⑧ 표 캡션 아래 몇 줄 안에서 출처 줄을 찾는가

    def __post_init__(self):
        if self.front_section < 0 or self.body_section < 0:
            raise ValueError("섹션 번호는 0 이상이어야 합니다")
        if self.front_section == self.body_section:
            raise ValueError("현재 파서는 차례·개요와 본문이 별도 섹션인 양식을 지원합니다")
        if not 0 < self.goal_similarity <= 1 or not 0 <= self.figtab_min_agreement <= 1:
            raise ValueError("유사도·일치율은 0~1 사이여야 합니다")
        for name in ("goal_lookahead", "source_lookback", "source_lookahead"):
            if getattr(self, name) < 1:
                raise ValueError(f"{name}은 1 이상이어야 합니다")

    def toc_config(self) -> TocConfig:
        return TocConfig(front_section=self.front_section, body_section=self.body_section)


DEFAULT_CONFIG = CheckConfig()
''')


In [ ]:
# 모듈: ncs_knowledge/figtab.py
write_module('ncs_knowledge/figtab.py', r'''"""F02 필요지식-⑦ 그림·표 번호 연속성 검사 (figtab v1.4 논리 그대로, ncs_core 어휘로 옮김).

check_figtab(sections, lesson_spans) -> findings: item_id / region_path / verdict / evidence_quote / reason / suggestion
기준
  - 학습 N 안의 그림은 [그림 N-1], [그림 N-2] … 순서대로, 표는 〈표 N-1〉, 〈표 N-2〉 … 순서대로
  - 괄호: 그림은 [ ], 표는 〈 〉
  - 본문에서 참조한 번호([그림 2-3]과 같이 …)는 실제 캡션이 있어야 함
"""
import re

from ncs_core import FAIL, PASS, PUA, UNDECIDED, visible

ITEM = 'F02-필요지식-⑦'

def pua_finding(item_id, rp, txt):
    if not PUA.search(txt or ''): return None
    return {'item_id': item_id + '-문자', 'region_path': rp, 'verdict': FAIL, 'evidence_quote': visible(txt),
            'reason': '기호 글꼴(사용자 정의 영역) 문자가 들어 있음 — 한글 화면에서는 가운뎃점 등으로 보이지만 다른 환경에서는 빈칸·네모로 깨짐',
            'suggestion': PUA.sub('·', txt)}

_FW = str.maketrans('０１２３４５６７８９－', '0123456789-')          # 전각 숫자·대시 → 반각
_DASH = r'[-–—－‐]'
LABEL = re.compile(rf'([\[〔\(（〈〈<＜《])\s*(그림|표)\s*(\d+)\s*{_DASH}\s*(\d+)\s*([\]〕\)）〉〉>＞》])')
BARE  = re.compile(rf'(그림|표)\s*(\d+)\s*{_DASH}\s*(\d+)')                # 괄호 없이 나온 번호(기호글꼴 괄호 의심)
CORRECT = {'그림': ('[', ']'), '표': ('〈', '〉')}
_OK_BRACKETS = {'그림': {('[', ']')}, '표': {('〈', '〉'), ('〈', '〉')}}  # U+3008/3009, U+2329/232A 둘 다 허용

# 학습 제목줄: "학습 1" / "학습 1. 제목" / "학습 1. 제목 12"(차례) / "학습 1 ○○하기"
HEAD = re.compile(r'^학습\s*(\d+)(?:\s*\.\s*(.*?)|\s*(\S.*?하기))?(?:\s+\d+)?$')
# 마지막 학습의 끝(참고자료 등)으로 볼 줄
TAIL = re.compile(r'^(교수\s*[·ㆍ]?\s*학습\s*)?참고\s*(자료|문헌)$|^활용\s*서식$|^부록$')

SENTENCE_END = re.compile(r'(다|요|라|오|죠|까)\s*[.。?]\s*$')
# 라벨 바로 뒤(띄어쓰기 없이) 조사가 붙으면 참조: "[그림 2-9]를", "〈표 1-1〉과 같이"
JOSA_AFTER = re.compile(r'^(은|는|이|가|을|를|과|와|의|에|로|처럼|같이|에서|부터|까지|도|만)(\s|$)')


def _flatten(sections):
    return [(s, i, t) for s, sec in enumerate(sections) for i, t in enumerate(sec) if t]


def split_lessons(flat, caps=None):
    """학습 제목줄로 본문을 학습별 구간으로 나눔. 반환 ([(학습번호, start, end)], 일치율)
    차례(section0)에도 '학습 1. …' 줄이 있으므로 1부터 연속하는 제목줄 묶음(run)이 여럿 나온다.
    caps가 있으면 '캡션의 앞 번호 == 그 캡션이 속한 학습 번호'인 비율이 가장 높은 run을 고르고(동률이면 넓은 것),
    없으면 가장 넓게 펼쳐진 run을 고른다. 일치율은 호출자가 경계 신뢰도 판단에 쓴다."""
    runs, cur = [], []
    for k, (s, i, t) in enumerate(flat):
        m = HEAD.match(t)
        if not m: continue
        no = int(m.group(1))
        if cur and no == cur[-1][1]: continue             # 같은 번호 반복(표지 '학습 1' + '학습 1. 제목') 무시
        if no == 1:
            if cur: runs.append(cur)
            cur = [(k, no)]
        elif cur and no == cur[-1][1] + 1:
            cur.append((k, no))
    if cur: runs.append(cur)
    if not runs: return [], 0.0

    def spans_of(run):
        out = []
        for j, (k, no) in enumerate(run):
            end = run[j + 1][0] if j + 1 < len(run) else \
                  next((kk for kk in range(k + 1, len(flat)) if TAIL.match(flat[kk][2])), len(flat))
            out.append((no, k, end))
        return out

    def agreement(spans):
        if not caps: return None
        hit = sum(1 for c in caps if any(a <= c['k'] < b and c['major'] == no for no, a, b in spans))
        return hit / len(caps)

    cands = [(spans_of(r), r) for r in runs]
    if caps:
        best = max(cands, key=lambda sr: (agreement(sr[0]), sr[1][-1][0] - sr[1][0][0], sr[1][0][0]))
    else:
        best = max(cands, key=lambda sr: (sr[1][-1][0] - sr[1][0][0], sr[1][0][0]))
    spans = best[0]
    return spans, (agreement(spans) if caps else 1.0)


def _fmt(kind, major, minor):
    o, c = CORRECT[kind]
    return f'{o}{kind} {major}-{minor}{c}'


def _f(item, rp, verdict, quote, reason, suggestion=None):
    return {'item_id': item, 'region_path': rp, 'verdict': verdict,
            'evidence_quote': visible(quote) if quote else quote, 'reason': reason, 'suggestion': suggestion}


def collect_labels(flat):
    """캡션(문단 첫머리의 라벨)과 참조(문장 중간의 라벨)를 나눠 모음."""
    caps, refs, bare = [], [], []
    for k, (s, i, t) in enumerate(flat):
        tt = t.translate(_FW)
        covered = []
        for j, m in enumerate(LABEL.finditer(tt)):
            covered.append(m.span())
            tail = tt[m.end():]
            rest = tail.strip()
            d = dict(kind=m.group(2), major=int(m.group(3)), minor=int(m.group(4)),
                     open=m.group(1), close=m.group(5), label=m.group(0), k=k, sec=s, para=i, text=t)
            # 캡션 판정: 문단이 라벨로 시작 + 라벨에 조사가 붙지 않음 + 뒤가 짧은 제목(문장으로 끝나지 않음)
            is_cap = (j == 0 and m.start() == 0 and len(rest) <= 80
                      and not JOSA_AFTER.match(tail) and not SENTENCE_END.search(rest))
            (caps if is_cap else refs).append(d)
        for m in BARE.finditer(tt):
            if not any(a <= m.start() < b for a, b in covered):
                bare.append(dict(kind=m.group(1), major=int(m.group(2)), minor=int(m.group(3)),
                                 label=m.group(0), k=k, sec=s, para=i, text=t))
    return caps, refs, bare


MIN_AGREEMENT = 0.5     # 캡션 앞 번호와 학습 경계가 이 비율 미만으로 맞으면 '-위치' 검사를 하지 않음


def _title(c):
    """라벨을 뺀 캡션 글(그림명). 공백은 로더가 이미 한 칸으로 줄였으므로 그대로 비교"""
    t = c['text'].translate(_FW)
    return t[len(c['label']):].strip() if t.startswith(c['label']) else t.strip()


def toc_lesson_count(sections):
    """차례(section0)의 '학습 N. …' 줄 개수 — toc_check의 parse_manuscript 와 같은 규칙"""
    n = 0
    for s in sections[0] if sections else []:
        m = re.match(r'학습\s*(\d+)\.\s*(.+)', s)
        if m and int(m.group(1)) == n + 1: n += 1
    return n


def check_figtab(sections, lesson_spans=None, *, min_agreement=MIN_AGREEMENT, labels=None):
    """lesson_spans: [(학습번호, flat_start, flat_end)] 를 밖(내비게이터 기반 학습 구간)에서 주면 그것을 씀.
    1차: 캡션 번호 자체로 연속성 검사(확정적). 2차: 앞 번호 열이 차례 학습 개수와 맞는가.
    3차: 학습 제목줄이 믿을 만하게 잡힐 때만, 캡션이 놓인 학습과 앞 번호가 같은가(-위치)."""
    if labels is None:
        flat = _flatten(sections)
        caps, refs, bare = collect_labels(flat)
    else:
        flat, caps, refs, bare = labels["flat"], labels["caps"], labels["refs"], labels["bare"]
    F = []
    n_toc = toc_lesson_count(sections)

    # ---------- 1차·2차: 종류별로 문서 순서대로 ----------
    for kind in ('그림', '표'):
        cs = [c for c in caps if c['kind'] == kind]
        if not cs: continue
        groups, g = [], []
        for c in cs:
            if g and c['major'] != g[-1]['major']: groups.append(g); g = []
            g.append(c)
        if g: groups.append(g)

        # 앞 번호 열: 1,2,3… 한 번씩 순서대로여야 함
        majors = [g[0]['major'] for g in groups]
        seen = []
        for g in groups:
            major = g[0]['major']; first = g[0]
            rp0 = f"section{first['sec']} > {kind} {major}-x 묶음"
            if major in seen:
                F.append(_f(ITEM + '-번호', rp0, FAIL, ' / '.join(c['label'] for c in g[:4]),
                            f'{kind} {major}-x 가 다른 번호 묶음 뒤에 다시 나옴(앞서 {kind} {max(seen)}-x 까지 나온 뒤). '
                            f'학습 {major}의 {kind}은 한 곳에 이어져야 함 — 뒤에 나온 {first["label"]} 부터는 놓인 학습에 맞춰 번호를 바꿀 것'))
            elif seen and major != seen[-1] + 1:
                F.append(_f(ITEM + '-번호', rp0, FAIL, ' / '.join(c['label'] for c in g[:4]),
                            f'{kind} 앞 번호가 {seen[-1]} 다음에 {major} 로 감 — 학습 순서대로 {seen[-1] + 1} 이어야 함'
                            + ('' if major > seen[-1] else '(되돌아감)')))
            elif not seen and major != 1:
                F.append(_f(ITEM + '-번호', rp0, FAIL, ' / '.join(c['label'] for c in g[:4]),
                            f'원고에서 처음 나오는 {kind}의 앞 번호가 {major} — 학습 1의 {kind}이면 {_fmt(kind, 1, 1)} 부터여야 함'))
            seen.append(major)

            # 뒷 번호: 묶음 안에서 1부터 연속
            bad, expect = 0, 1
            for nth, c in enumerate(g, 1):
                rp = f"section{c['sec']} > {c['label']}"
                if (pf := pua_finding(ITEM, rp, c['text'])): F.append(pf)
                same = None
                if c['minor'] == expect:
                    expect += 1
                else:
                    bad += 1
                    if c['minor'] < expect:
                        prev = next((p for p in g[:nth - 1] if p['minor'] == c['minor']), None)
                        if prev and _title(prev) == _title(c):
                            same = prev
                            F.append(_f(ITEM + '-번호', rp, FAIL, c['text'],
                                        f'{c["label"]} 이(가) 두 번 나오고 그림명도 "{_title(c)}" 로 똑같음 — 같은 {kind}을 두 번 넣은 것. '
                                        f'하나를 빼면 뒤 번호는 그대로 맞음', '(중복이면 삭제)'))
                        elif prev:
                            F.append(_f(ITEM + '-번호', rp, FAIL, c['text'],
                                        f'{c["label"]} 이(가) 두 번 나오는데 그림명이 다름 — 앞: "{_title(prev)}", 뒤: "{_title(c)}". '
                                        f'다른 {kind}이므로 뒤의 것은 {_fmt(kind, major, expect)} 이어야 함', _fmt(kind, major, expect)))
                        else:
                            F.append(_f(ITEM + '-번호', rp, FAIL, c['text'],
                                        f'{c["label"]} 이(가) {_fmt(kind, major, expect - 1)} 뒤에 나옴 — 순서가 뒤바뀜. '
                                        f'자리를 옮기거나, 이 자리라면 {_fmt(kind, major, expect)} 이어야 함', _fmt(kind, major, expect)))
                    else:
                        skipped = ', '.join(_fmt(kind, major, m) for m in range(expect, c['minor']))
                        F.append(_f(ITEM + '-번호', rp, FAIL, c['text'],
                                    f'{kind} {major}-x 묶음의 {nth}번째인데 {c["label"]} 로 매겨짐 — {skipped} 를 건너뜀. {_fmt(kind, major, expect)} 이어야 함',
                                    _fmt(kind, major, expect)))
                    if not same: expect = max(expect, c['minor']) + 1   # 한 번 어긋난 뒤 연쇄 보고 방지
                if (c['open'], c['close']) not in _OK_BRACKETS[kind]:
                    F.append(_f(ITEM + '-표기', rp, FAIL, c['text'],
                                f"{kind} 번호는 {CORRECT[kind][0]} {CORRECT[kind][1]} 로 묶어야 하는데 {c['open']} {c['close']} 로 묶음",
                                _fmt(kind, c['major'], c['minor'])))
            if bad == 0:
                seq = ', '.join(c['label'] for c in g) if len(g) <= 4 else f"{g[0]['label']} … {g[-1]['label']}"
                F.append(_f(ITEM, f'{kind} {major}-x', PASS, ' / '.join(c['label'] for c in g),
                            f'{kind} {major}-x {len(g)}개가 {seq} 순서로 빠짐없이 이어짐'))

        # 차례의 학습 개수와 대조
        if n_toc and majors and max(majors) > n_toc:
            F.append(_f(ITEM + '-번호', f'{kind} 앞 번호', FAIL, ', '.join(f'{kind} {m}-x' for m in majors),
                        f'{kind} 앞 번호가 {max(majors)} 까지 나오는데 차례의 학습은 {n_toc}개 — 학습 번호를 넘는 번호가 있음'))

    # ---------- 3차: 학습 제목줄이 믿을 만할 때만 '놓인 학습' 과 대조 ----------
    spans, agree = (lesson_spans, 1.0) if lesson_spans else split_lessons(flat, caps)
    if spans and caps and agree >= min_agreement:
        lesson_of = lambda k: next((no for no, a, b in spans if a <= k < b), None)
        for c in caps:
            no = lesson_of(c['k'])
            if no is not None and c['major'] != no:
                F.append(_f(ITEM + '-위치', f"section{c['sec']} > 학습 {no} > {c['label']}", FAIL, c['text'],
                            f'{c["label"]} 이(가) 학습 {no} 본문(제목줄 "{flat[[a for n,a,b in spans if n==no][0]][2][:20]}" 아래)에 놓여 있음 — '
                            f'앞 번호 {c["major"]} 와 다름. 학습 {no}에 두려면 {c["kind"]} {no}-x 로',
                            None))
    elif caps:
        F.append(_f(ITEM + '-위치', '본문', UNDECIDED, None,
                    '학습 제목줄을 믿을 만하게 잡지 못해(캡션-경계 일치율 '
                    f'{agree:.0%}) "캡션이 놓인 학습 = 앞 번호" 검사는 못 함. 번호 연속성 검사(위)는 영향 없음'))

    # ---------- 참조 · 괄호 없음 ----------
    cap_keys = {(c['kind'], c['major'], c['minor']) for c in caps}
    for r in refs:
        rp = f"section{r['sec']} > 본문"
        if (r['kind'], r['major'], r['minor']) not in cap_keys:
            F.append(_f(ITEM + '-참조', rp, FAIL, r['text'],
                        f'본문 문장이 {r["label"]} 을(를) 가리키지만 원고 어디에도 {r["label"]} 캡션이 없음 — 그림·표를 빼거나 번호를 바꾼 뒤 본문을 안 고쳤을 가능성'))
        if (r['open'], r['close']) not in _OK_BRACKETS[r['kind']]:
            F.append(_f(ITEM + '-표기', rp, FAIL, r['text'],
                        f"본문에서 {r['kind']} 번호를 {r['open']} {r['close']} 로 묶음 — {CORRECT[r['kind']][0]} {CORRECT[r['kind']][1]} 로 묶어야 함",
                        _fmt(r['kind'], r['major'], r['minor'])))
    for x in bare:
        F.append(_f(ITEM + '-표기', f"section{x['sec']} > 본문", UNDECIDED, x['text'],
                    f'"{x["label"]}" 이 괄호 없이 나옴 — 괄호가 기호 글꼴(사용자 정의 문자)로 찍혔거나 빠진 것. 어느 쪽인지 원고에서 확인',
                    _fmt(x['kind'], x['major'], x['minor'])))
    return F
''')


In [ ]:
# 모듈: ncs_knowledge/models.py
write_module('ncs_knowledge/models.py', r'''"""파서와 규칙 사이의 경계. 결과 모델(ReviewResult 등)은 ncs_core.models를 쓴다."""

from dataclasses import dataclass
from typing import Any

from ncs_core import Document

from .config import CheckConfig


@dataclass(frozen=True)
class ReviewContext:
    manuscript: Document
    plan: Document | None
    module_code: str
    config: CheckConfig
    toc: dict[str, Any]                 # ncs_toc.parsers.parse_toc
    overview: dict[str, Any] | None     # ncs_overview.parsers.parse_overview 의 '내용체계' (표 기준)
    body: dict[str, Any]                # ncs_toc.parsers.parse_body
    units: list[dict[str, Any]]         # structure.unit_regions — 학습 내용(N-M) 구간
    plan_goals: dict[str, Any] | None   # plan.parse_plan_goals — 계획서 학습 내용별 학습 목표
    table_paragraphs: set[int] | None   # 본문 섹션에서 표 셀 안에 든 문단 인덱스 (hwpx만, hwp는 None)
    labels: dict[str, Any]              # figtab.collect_labels — flat, caps, refs, bare, lesson_spans
''')


In [ ]:
# 모듈: ncs_knowledge/pipeline.py
write_module('ncs_knowledge/pipeline.py', r'''"""입력 선택 → 추출 → 규칙 실행. 저장과 화면 출력은 호출자가 결정한다. (ncs_toc.pipeline과 같은 구조)

추출은 기존 모듈을 그대로 쓴다:
  차례·본문·참고 자료 위치: ncs_toc.parsers / ncs_toc.presence
  내용체계 표:             ncs_overview.parsers.parse_overview
  계획서 학습 목표:        ncs_knowledge.plan
  그림·표 캡션:            ncs_knowledge.figtab.collect_labels
"""

import re
from collections.abc import Sequence
from typing import Any

from ncs_core import Document, ReviewResult, Rule, resolve_module_code, run_rules, select_documents
from ncs_core.text import flatten
from ncs_overview.parsers import parse_overview
from ncs_toc.parsers import parse_body, parse_toc
from ncs_toc.presence import section_presence

from .config import DEFAULT_CONFIG, CheckConfig
from .figtab import _flatten, collect_labels
from .models import ReviewContext
from .plan import parse_plan_goals
from .registry import ALL_ITEM_IDS, DEFAULT_RULES
from .structure import cell_paragraphs, unit_regions


def prepare_review(
    manuscript: Document,
    plan: Document | None = None,
    *,
    module_code: str = "",
    config: CheckConfig = DEFAULT_CONFIG,
) -> ReviewContext:
    code = resolve_module_code(manuscript.path, module_code)
    if max(config.front_section, config.body_section) >= len(manuscript.sections):
        raise ValueError("지정한 차례/본문 섹션이 없습니다. 문서 구조와 CheckConfig를 확인하세요")
    toc_config = config.toc_config()
    sections = [manuscript.sections[config.front_section], manuscript.sections[config.body_section]]
    toc = parse_toc(sections)
    if not toc or not toc["학습"]:
        raise ValueError("차례 또는 차례 학습명을 읽지 못했습니다. 양식과 섹션 설정을 확인하세요")
    overview = parse_overview(manuscript, config.front_section).get("내용체계")
    images = [(p, name) for s, p, name in manuscript.images if s == config.body_section]
    presence = section_presence(sections, images, toc_config)
    body = parse_body(sections, len(toc["학습"]), cutoff=presence["참고 자료"]["k"], config=toc_config)
    units = unit_regions(body, config)
    plan_goals = parse_plan_goals(plan, code) if plan else None
    table_paragraphs = cell_paragraphs(manuscript.path, config.body_section) if manuscript.format == "hwpx" else None
    flat = _flatten(sections)
    caps, refs, bare = collect_labels(flat)
    position = {(s, i): k for k, (s, i, _) in enumerate(flat)}
    lesson_spans = [
        (no, position[(1, body["lines"][a][0])], position[(1, body["lines"][b][0])] if b < len(body["lines"]) else len(flat))
        for no, a, b in body["spans"]
    ]
    labels = {"flat": flat, "caps": caps, "refs": refs, "bare": bare, "lesson_spans": lesson_spans}
    return ReviewContext(manuscript, plan, code, config, toc, overview, body, units, plan_goals, table_paragraphs, labels)


def _actual_sections(value: Any, config: CheckConfig) -> Any:
    if isinstance(value, str):
        return re.sub(r"section([01])\b", lambda m: "section" + str(config.front_section if m[1] == "0" else config.body_section), value)
    if isinstance(value, dict):
        return {k: _actual_sections(v, config) for k, v in value.items()}
    if isinstance(value, list):
        return [_actual_sections(v, config) for v in value]
    return value


def _diagnostics(context: ReviewContext, rules: Sequence[Rule]) -> dict:
    manuscript, plan, config = context.manuscript, context.plan, context.config
    warnings = []
    if not plan:
        warnings.append("개발계획서가 없어 ⑤ 학습 목표 대조는 생략(-)입니다")
    elif not context.plan_goals:
        warnings.append("개발계획서에서 이 모듈 코드의 블록을 찾지 못했습니다. ⑤는 생략(-)입니다")
    elif not context.plan_goals["내용"]:
        warnings.append("개발계획서 학습 내용·학습 목표를 읽지 못했습니다")
    if not context.body["nav_runs"]:
        warnings.append("내비게이터 묶음을 찾지 못했습니다. ①②③은 판단불가, 학습 구간이 필요한 ④⑤⑥⑦도 영향을 받습니다")
    if context.overview is None:
        warnings.append("'학습모듈의 내용체계' 표를 읽지 못했습니다. ③④의 내용체계 대조는 판단불가입니다")
    if context.table_paragraphs is None:
        warnings.append("hwp 원고는 표 안 문단을 구분하지 않아 ⑥⑧에서 표 안 번호·출처도 함께 셉니다")
    if len(manuscript.sections) > 2:
        warnings.append("선택한 차례/개요 및 본문 섹션만 검사합니다")
    implemented = {rule.item_prefix for rule in rules}
    implemented_items = [item for item in ALL_ITEM_IDS if any(item.startswith(p) for p in implemented)]
    labels = context.labels
    return {
        "module_code": context.module_code,
        "inputs": {"manuscript": str(manuscript.path), "plan": str(plan.path) if plan else None},
        "format": manuscript.format,
        "sections": {"front": config.front_section, "body": config.body_section},
        "counts": {
            "lessons": len(context.toc["학습"]),
            "contents": len(context.toc["내용"]),
            "navigator_runs": len(context.body["nav_runs"]),
            "units": len(context.units),
            "plan_goal_sentences": sum(len(v["goals"]) for v in context.plan_goals["내용"].values()) if context.plan_goals else 0,
            "captions": len(labels["caps"]),
            "references": len(labels["refs"]),
            "bare_labels": len(labels["bare"]),
            "table_paragraphs": len(context.table_paragraphs) if context.table_paragraphs is not None else None,
        },
        "implemented": implemented_items,
        "not_implemented": [item for item in ALL_ITEM_IDS if item not in implemented_items],
        "warnings": warnings,
    }


def evaluate_review(context: ReviewContext, *, rules: Sequence[Rule] = DEFAULT_RULES) -> ReviewResult:
    config = context.config
    checklist = run_rules(context, rules)
    for row in checklist:
        row["region_path"] = _actual_sections(row["region_path"], config)
        row["비고"] = _actual_sections(row["비고"], config)
    diagnostics = _diagnostics(context, rules)
    return ReviewResult(context.module_code, checklist, flatten(checklist), diagnostics, context)


def run_review(
    paths,
    *,
    roles: dict | None = None,
    module_code: str = "",
    config: CheckConfig = DEFAULT_CONFIG,
    rules: Sequence[Rule] = DEFAULT_RULES,
) -> ReviewResult:
    documents = select_documents(paths, roles, required=("manuscript",))
    manuscript, plan = documents["manuscript"], documents.get("plan")
    context = prepare_review(manuscript, plan, module_code=module_code, config=config)
    return evaluate_review(context, rules=rules)
''')


In [ ]:
# 모듈: ncs_knowledge/plan.py
write_module('ncs_knowledge/plan.py', r'''"""개발계획서 모듈 블록에서 학습 내용별 학습 목표(수행 준거) 문장을 읽는다.

블록 기준은 ncs_overview.parsers.parse_plan과 같다. 그 파서는 학습명·내용명·요소만 읽으므로 여기서는 목표 문장을 더 읽는다.
"""

import re

from ncs_core import Document
from ncs_overview.patterns import ELEMENT_CODE, OV_SUB, PLAN_LESSON

GOAL_SENTENCE = re.compile(r"^\d+\.\d+\s+\S")  # '1.1 호텔의 …할 수 있다.'


def parse_plan_goals(document: Document, module_code: str) -> dict | None:
    """→ {'학습명': [...], '요소명': [...], '내용': {(N, M): {'text', 'goals': [...]}}}. 블록을 못 찾으면 None."""
    lines = [l for section in document.sections for l in section]
    code = module_code.replace("LM", "")
    start = next(
        (i for i, l in enumerate(lines) if l == code and i > 0 and lines[i - 1].startswith("능력단위 코드번호")),
        None,
    )
    if start is None:
        return None
    end = next(
        (i for i in range(start + 1, len(lines)) if lines[i].startswith("NCS (수정 금지)") or lines[i].startswith("능력단위 코드번호")),
        len(lines),
    )
    block = [l for l in lines[start:end] if l]
    plan, current, i = {"학습명": [], "요소명": [], "내용": {}}, None, 0
    while i < len(block):
        line = block[i]
        if (m := ELEMENT_CODE.match(line)) and m.group(1) == code and i + 1 < len(block):
            plan["요소명"].append(block[i + 1])
            current = None
            i += 2
            continue
        if m := OV_SUB.match(line):
            text = m.group(3).strip()
            if not text and i + 1 < len(block):
                text = block[i + 1]
                i += 1
            current = (int(m.group(1)), int(m.group(2)))
            plan["내용"][current] = {"text": text, "goals": []}
        elif PLAN_LESSON.match(line):
            plan["학습명"].append(line)
            current = None
        elif current and GOAL_SENTENCE.match(line):
            plan["내용"][current]["goals"].append(line)
        i += 1
    return plan
''')


In [ ]:
# 모듈: ncs_knowledge/registry.py
write_module('ncs_knowledge/registry.py', r'''"""규칙 추가 지점. 규칙은 ReviewContext 하나를 받아 기존 형식의 행 목록을 돌려준다."""

from ncs_core import Rule

from .rules.contents import check_contents
from .rules.figures import check_figures
from .rules.goals import check_goals
from .rules.navigator import check_navigator
from .rules.numbering import check_numbering
from .rules.sources import check_sources


def _sections(c):
    return [c.manuscript.sections[c.config.front_section], c.manuscript.sections[c.config.body_section]]


DEFAULT_RULES = (
    # ①②③은 같은 내비게이터 묶음을 한 번 읽어 세 항목의 행을 함께 낸다.
    Rule("navigator", "F02-필요지식-", lambda c: check_navigator(c.toc, c.overview, c.body)),
    Rule("contents", "F02-필요지식-④", lambda c: check_contents(c.toc, c.overview, c.body, c.units)),
    Rule("goals", "F02-필요지식-⑤", lambda c: check_goals(c.plan_goals, c.units, similarity=c.config.goal_similarity, whitespace_is_difference=c.config.whitespace_is_difference)),
    Rule("numbering", "F02-필요지식-⑥", lambda c: check_numbering(c.body, c.units, c.table_paragraphs)),
    Rule("figures", "F02-필요지식-⑦", lambda c: check_figures(_sections(c), c.labels, min_agreement=c.config.figtab_min_agreement)),
    Rule("sources", "F02-필요지식-⑧", lambda c: check_sources(_sections(c)[1], 1, c.labels["caps"], c.table_paragraphs, lookback=c.config.source_lookback, lookahead=c.config.source_lookahead)),
)

ALL_ITEM_IDS = tuple("F02-필요지식-" + n for n in "①②③④⑤⑥⑦⑧")
''')


In [ ]:
# 모듈: ncs_knowledge/reporting.py
write_module('ncs_knowledge/reporting.py', r'''"""필요 지식 검수 결과 저장. 파일 이름 규칙은 ncs_core.reporting과 같다 (v1.0 노트북의 꼬리말 유지)."""

from ncs_core import ReviewResult, SavedReport, save_report

ITEM_FILES = (
    ("F02-필요지식-①", "필요지식①_내비게이터_코드번호"),
    ("F02-필요지식-②", "필요지식②_내비게이터_하기종결"),
    ("F02-필요지식-③", "필요지식③_내비게이터_차례내용체계일치"),
    ("F02-필요지식-④", "필요지식④_학습내용명_차례내용체계일치"),
    ("F02-필요지식-⑤", "필요지식⑤_학습목표_개발계획서일치"),
    ("F02-필요지식-⑥", "필요지식⑥_번호체계"),
    ("F02-필요지식-⑦", "필요지식⑦_그림표번호"),
    ("F02-필요지식-⑧", "필요지식⑧_출처표기"),
)


def save_results(result: ReviewResult, out_dir="결과", *, per_run_dir: bool = True) -> SavedReport:
    return save_report(
        result, out_dir, name="필요지식", item_files=ITEM_FILES, other_tail="필요지식_기타", per_run_dir=per_run_dir
    )
''')


In [ ]:
# 모듈: ncs_knowledge/rules/__init__.py
write_module('ncs_knowledge/rules/__init__.py', r'''"""문서 읽기와 독립적인 필요 지식 검수 규칙."""
''')


In [ ]:
# 모듈: ncs_knowledge/rules/common.py
write_module('ncs_knowledge/rules/common.py', r'''"""규칙 모듈이 함께 쓰는 행 생성기."""

from ncs_core import PASS, verdict

ITEM = "F02-필요지식-"


def row(no: str, region_path: str, source, checks: dict, notes: dict, suggestion=None, **extra) -> dict:
    result = verdict(checks)
    out = {
        "item_id": ITEM + no,
        "region_path": region_path,
        "원고": source,
        "체크": checks,
        "판정": result,
        "비고": notes,
        "수정 제안": None if result == PASS else suggestion,
    }
    out.update(extra)
    return out
''')


In [ ]:
# 모듈: ncs_knowledge/rules/contents.py
write_module('ncs_knowledge/rules/contents.py', r'''"""F02 필요지식 ④: 본문 학습 내용명(N-M.) = 차례 면 = 내용체계 표, N = 놓인 학습, M 연번."""

from ncs_core import O, PUA, TRI, X, visible
from ncs_toc.compare import diff_list

from .common import row


def check_contents(toc: dict, overview: dict | None, body: dict, units: list[dict]) -> list[dict]:
    rows = []
    for no, a, b in body["spans"]:
        own = [u for u in units if u["lesson"] == no]
        region = f"section1 > 학습 {no} > N-M. 제목"
        actual = [u["text"] for u in own]
        checks, notes = {}, {}
        for name, ref in (
            ("차례 학습 내용명과 일치", [s["text"] for s in toc["내용"] if s["no"] == no]),
            ("내용체계 학습 내용명과 일치", [s["text"] for s in overview["내용"] if s["no"] == no] if overview else None),
        ):
            if ref is None:
                checks[name], notes[name] = TRI, "'학습모듈의 내용체계' 표를 찾지 못함"
                continue
            value, diff = diff_list("학습 내용명", ref, actual)
            checks[name] = value
            if diff:
                notes[name] = " / ".join(diff)
        wrong = [u for u in own if u["no"] != no]
        checks["N-M 의 N = 놓인 학습"] = X if wrong else O
        if wrong:
            notes["N-M 의 N = 놓인 학습"] = f"학습 {no}에 놓였는데 번호가 다름: " + ", ".join(f"{u['no']}-{u['sub']}. {visible(u['text'])}" for u in wrong)
        subs_no = [u["sub"] for u in own]
        checks["M 연번"] = O if subs_no == list(range(1, len(own) + 1)) else X
        if checks["M 연번"] == X:
            notes["M 연번"] = f"본문 순서 {subs_no} — 1부터 연속이어야 함"
        pua = [u["text"] for u in own if PUA.search(u["text"])]
        checks["기호 글꼴 문자 없음"] = X if pua else O
        if pua:
            notes["기호 글꼴 문자 없음"] = "기호 글꼴 문자: " + ", ".join(visible(t) for t in pua)
        rows.append(
            row("④", region, [f"{u['no']}-{u['sub']}. {visible(u['text'])}" for u in own], checks, notes,
                "차례·내용체계 값으로 통일", 기준={"차례": [s["text"] for s in toc["내용"] if s["no"] == no]})
        )
    return rows
''')


In [ ]:
# 모듈: ncs_knowledge/rules/figures.py
write_module('ncs_knowledge/rules/figures.py', r'''"""F02 필요지식 ⑦: 그림·표 번호 연속 — figtab 검사 결과(findings)를 체크리스트 행으로 바꾼다."""

from ncs_core import FAIL, O, PASS, TRI, UNDECIDED, X

from ..figtab import ITEM as FIGTAB_ITEM
from ..figtab import check_figtab
from .common import ITEM

VALUE = {PASS: O, FAIL: X, UNDECIDED: TRI}


def check_figures(sections, labels: dict, *, min_agreement: float) -> list[dict]:
    rows = []
    for finding in check_figtab(sections, labels["lesson_spans"], min_agreement=min_agreement, labels=labels):
        sub = finding["item_id"].replace(FIGTAB_ITEM, "").lstrip("-") or "번호 연속(묶음)"
        value = VALUE[finding["verdict"]]
        rows.append(
            {
                "item_id": ITEM + "⑦",
                "region_path": finding["region_path"],
                "원고": finding.get("evidence_quote"),
                "체크": {sub: value},
                "판정": finding["verdict"],
                "비고": {sub: finding["reason"]} if value != O else {},
                "수정 제안": finding.get("suggestion"),
            }
        )
    return rows
''')


In [ ]:
# 모듈: ncs_knowledge/rules/goals.py
write_module('ncs_knowledge/rules/goals.py', r'''"""F02 필요지식 ⑤: 각 학습 내용의 '학습 목표' = 개발계획서 학습 목표(수행 준거).

같음 O / 공백·문장부호만 다름 X(설정으로 O+참고) / 유사도 ≥ 임계값 윤문 △ / 그 밖·누락·본문에만 있음 X.
"""

import difflib
import re

from ncs_core import NA, O, PUA, TRI, X, visible

from .common import row


def goal_norm(text: str) -> str:
    text = re.sub(r"^\s*(\d+\.\d+|[•·▪ㆍ‧◦○\-])\s*", "", text or "")  # '1.1 ' / '•' 제거
    text = PUA.sub("·", text)
    return re.sub(r"\s+", " ", text).strip().rstrip(".")


def goal_key(text: str) -> str:
    """공백·부호·대소문자를 뺀 비교 키 (윤문 유사도 계산용)."""
    return re.sub(r"[\s·,.()\[\]/\-]", "", goal_norm(text)).lower()


def check_goals(plan: dict | None, units: list[dict], *, similarity: float, whitespace_is_difference: bool) -> list[dict]:
    rows = []
    for unit in units:
        key = f"{unit['no']}-{unit['sub']}"
        region = f"section1 > 학습 {unit['lesson']} > {key}. {visible(unit['text'])} > 학습 목표"
        actual = unit["goal"]
        if not plan:
            rows.append(row("⑤", region, [visible(a) for a in actual], {"개발계획서 학습 목표 일치": NA}, {"개발계획서": "개발계획서를 읽지 못해 대조 생략"}))
            continue
        checks, notes = {}, {}
        if unit["goal_k"] is None:
            checks["학습 목표 블록"], notes["학습 목표 블록"] = X, "제목 아래 '학습 목표' 줄을 찾지 못함"
        expected_unit = plan["내용"].get((unit["no"], unit["sub"]))
        if expected_unit is None:
            checks["개발계획서 학습 목표 일치"] = X
            notes["개발계획서 학습 목표 일치"] = f"개발계획서에 {key} 가 없음 (계획서 목록: {sorted(plan['내용'])})"
            rows.append(row("⑤", region, [visible(a) for a in actual], checks, notes))
            continue
        expected = expected_unit["goals"]
        checks["학습 목표 개수"] = O if len(expected) == len(actual) else X
        if checks["학습 목표 개수"] == X:
            notes["학습 목표 개수"] = f"본문 {len(actual)} / 개발계획서 {len(expected)}"
        exp_keys, act_keys = [goal_key(e) for e in expected], [goal_key(a) for a in actual]
        problems, soft = [], []
        for i, e in enumerate(expected):
            if i < len(actual) and goal_norm(e) == goal_norm(actual[i]):
                continue
            if exp_keys[i] in act_keys:  # 공백·문장부호만 다름
                j = act_keys.index(exp_keys[i])
                message = f'{i + 1}번째 공백·부호만 다름: 본문 "{visible(goal_norm(actual[j]))}" / 계획서 "{goal_norm(e)}"' + ("" if j == i else f" (본문 {j + 1}번째 자리)")
                (problems if whitespace_is_difference else soft).append((X, message))
                continue
            best = max(((difflib.SequenceMatcher(None, exp_keys[i], a, autojunk=False).ratio(), j) for j, a in enumerate(act_keys)), default=(0, None))
            if best[0] >= similarity:
                problems.append((TRI, f'{i + 1}번째 윤문 수준 차이(유사 {best[0]:.0%}): 본문 "{visible(goal_norm(actual[best[1]]))}" / 계획서 "{goal_norm(e)}" — 직능연 협의 여부 확인'))
            else:
                problems.append((X, f'{i + 1}번째 불일치 또는 누락: 계획서 "{goal_norm(e)}"'))
        extra = [
            a for j, a in enumerate(actual)
            if act_keys[j] not in exp_keys and not any(difflib.SequenceMatcher(None, act_keys[j], e, autojunk=False).ratio() >= similarity for e in exp_keys)
        ]
        for a in extra:
            problems.append((X, f'본문에만 있음(임의 추가 불가): "{visible(goal_norm(a))}"'))
        if problems:
            checks["개발계획서 학습 목표 일치"] = X if any(v == X for v, _ in problems) else TRI
            notes["개발계획서 학습 목표 일치"] = " / ".join(m for _, m in problems)
        else:
            checks["개발계획서 학습 목표 일치"] = O
        if soft:
            notes["개발계획서 학습 목표 일치(참고)"] = " / ".join(m for _, m in soft) + " — 통과 처리"
        rows.append(row("⑤", region, [visible(a) for a in actual], checks, notes, "개발계획서 학습 목표 문장으로 통일", 개발계획서=expected))
    return rows
''')


In [ ]:
# 모듈: ncs_knowledge/rules/navigator.py
write_module('ncs_knowledge/rules/navigator.py', r'''"""F02 필요지식 ①②③: 내비게이터 학습명 — 코드 번호 삭제 / '~하기' 종결 / 차례 면·내용체계 표 학습명과 일치."""

import re

from ncs_core import O, PUA, TRI, X, visible

from ..text import norm, same
from .common import row

CODE = re.compile(r"(LM\s?)?\d{10}_\d{2}v\d+(\.\d+)?|LM\s?\d{8,10}|\(\s*\d{10}[^)]*\)")  # 능력단위·요소·모듈 코드


def check_navigator(toc: dict, overview: dict | None, body: dict) -> list[dict]:
    rows = []
    runs = body["nav_runs"]
    if not runs:
        for no in ("①", "②", "③"):
            rows.append(row(no, "section1 > 내비게이터", None, {"내비게이터 찾기": TRI}, {"내비게이터 찾기": "'학습 1'…'학습 N' 제목 묶음을 찾지 못함"}))
        return rows
    n_toc = len(toc["학습"])
    if len(runs) != n_toc or any(len(r) != n_toc for r in runs):
        rows.append(
            row("③", "section1 > 내비게이터", f"묶음 {len(runs)}개, 길이 {[len(r) for r in runs]}",
                {"내비게이터 묶음 수·길이": TRI},
                {"내비게이터 묶음 수·길이": f"차례 학습 {n_toc}개 — 묶음 수나 학습 수가 다름(학습 구간이 어긋났을 수 있음)"})
        )
    for index in range(max(len(r) for r in runs)):
        no = index + 1
        texts = [(ri + 1, r[index]["text"]) for ri, r in enumerate(runs) if index < len(r)]
        region = f"section1 > 내비게이터 > 학습 {no} (묶음 {len(texts)}회)"
        first = texts[0][1]
        # ① 코드 번호
        hit = [(ri, t) for ri, t in texts if CODE.search(t)]
        checks = {"코드 번호 없음": X if hit else O}
        notes = {"코드 번호 없음": "코드가 남아 있음: " + " / ".join(f'묶음{ri} "{visible(t)}"' for ri, t in hit[:3])} if hit else {}
        rows.append(row("①", region, visible(first), checks, notes, CODE.sub("", first).strip() if hit else None))
        # ② '~하기'
        bad = [(ri, t) for ri, t in texts if not re.search(r"하기$", t.strip())]
        checks = {"'~하기' 종결": X if bad else O}
        notes = {"'~하기' 종결": "'~하기'로 끝나지 않음: " + " / ".join(f'묶음{ri} "{visible(t)}"' for ri, t in bad[:3])} if bad else {}
        rows.append(row("②", region, visible(first), checks, notes))
        # ③ 차례·내용체계 일치 + 묶음 간 일치 + 기호 글꼴
        checks, notes = {}, {}
        toc_text = toc["학습"][index]["text"] if index < n_toc else None
        ov_text = overview["학습"][index]["text"] if overview and index < len(overview["학습"]) else None
        for name, ref in (("차례 학습명과 일치", toc_text), ("내용체계 학습명과 일치", ov_text)):
            if ref is None:
                checks[name], notes[name] = TRI, "기준 쪽에 이 번호의 학습이 없음"
                continue
            diff = [(ri, t) for ri, t in texts if not same(ref, t)]
            checks[name] = O if not diff else X
            if diff:
                notes[name] = f'기준 "{visible(ref)}" / ' + " / ".join(f'묶음{ri} "{visible(t)}"' for ri, t in diff[:3])
        unique = {norm(t) for _, t in texts}
        checks["묶음 간 일치"] = O if len(unique) == 1 else X
        if len(unique) > 1:
            notes["묶음 간 일치"] = "내비게이터 묶음마다 학습명이 다름: " + " / ".join(sorted(visible(u) for u in unique))
        pua = [t for _, t in texts if PUA.search(t)]
        checks["기호 글꼴 문자 없음"] = X if pua else O
        if pua:
            notes["기호 글꼴 문자 없음"] = f"기호 글꼴 문자: {visible(pua[0])} — ·(U+00B7)로 교체"
        rows.append(
            row("③", region, visible(first), checks, notes,
                norm(toc_text) if toc_text and checks.get("차례 학습명과 일치") == X else None,
                기준={"차례": visible(toc_text) if toc_text else None, "내용체계": visible(ov_text) if ov_text else None})
        )
    return rows
''')


In [ ]:
# 모듈: ncs_knowledge/rules/numbering.py
write_module('ncs_knowledge/rules/numbering.py', r'''"""F02 필요지식 ⑥: 본문 번호 체계 󰊱 〉 1. 〉 (1) 〉 (가) 〉 1) 〉 가) 〉 ① 〉 ㉠ 〉 ⓐ (F11 p.43).

학습 내용(N-M)당 1행. 안에서 '필요 지식 / 수행 내용 / 수행 순서 / 재료 / 안전…' 머리말마다 번호가 새로 시작한다고 본다.
규칙 3개: 첫 수준이 󰊱 / 수준은 +1만(건너뜀 X) / 같은 수준은 연번.
텍스트 첫 글자로 수준을 판정하므로 한글 자동 번호(문단 번호 속성)는 보이지 않는다 → 번호 문단 0개면 △.
"""

import re

from ncs_core import O, TRI, X, visible

from ..structure import MEMO_LINE, mark
from .common import row

_GA = "가나다라마바사아자차카타파하"
LEVELS = [  # (수준, 이름, 정규식, 서수 함수)
    (1, "󰊱", re.compile(r"^([\U000F02B1-\U000F02C9]|[❶-❿➊-➓])"),
     lambda m: (ord(m.group(1)) - 0xF02B0) if ord(m.group(1)) >= 0xF02B1 else ((ord(m.group(1)) - 0x2775) if ord(m.group(1)) < 0x278A else ord(m.group(1)) - 0x2789)),
    (2, "1.", re.compile(r"^(\d{1,2})\.(?!\d)\s*"), lambda m: int(m.group(1))),
    (3, "(1)", re.compile(r"^\((\d{1,2})\)\s*"), lambda m: int(m.group(1))),
    (4, "(가)", re.compile(rf"^\(([{_GA}])\)\s*"), lambda m: _GA.index(m.group(1)) + 1),
    (5, "1)", re.compile(r"^(\d{1,2})\)\s*"), lambda m: int(m.group(1))),
    (6, "가)", re.compile(rf"^([{_GA}])\)\s*"), lambda m: _GA.index(m.group(1)) + 1),
    (7, "①", re.compile(r"^([①-⑳])"), lambda m: ord(m.group(1)) - 0x2460 + 1),
    (8, "㉠", re.compile(r"^([㉠-㉭])"), lambda m: ord(m.group(1)) - 0x3260 + 1),
    (9, "ⓐ", re.compile(r"^([ⓐ-ⓩ])"), lambda m: ord(m.group(1)) - 0x24D0 + 1),
]
NAMES = {level: name for level, name, _, _ in LEVELS}
NOT_A_NUMBER = (re.compile(r"^\d+-\d+\."), re.compile(r"^\d+\.\d+\s"), re.compile(r"^\[그림|^[〈<]표"))


def level_of(text: str):
    if any(p.match(text) for p in NOT_A_NUMBER):
        return None
    for level, _, pattern, ordinal in LEVELS:
        if m := pattern.match(text):
            return level, ordinal(m)
    return None


def check_numbering(body: dict, units: list[dict], table_paragraphs: set[int] | None) -> list[dict]:
    rows, lines = [], body["lines"]
    for unit in units:
        key = f"{unit['no']}-{unit['sub']}"
        region = f"section1 > 학습 {unit['lesson']} > {key}. {visible(unit['text'])}"
        segments, segment, name = [], [], "제목 아래"
        for k in range(unit["k0"] + 1, unit["k1"]):
            i, text = lines[k]
            if head := mark(text):
                segments.append((name, segment))
                segment, name = [], head
                continue
            if table_paragraphs is not None and i in table_paragraphs:
                continue
            if MEMO_LINE.match(text):
                continue
            if found := level_of(text):
                segment.append((i, found[0], found[1], text))
        segments.append((name, segment))
        checks = {"첫 수준이 󰊱": O, "수준 순서(건너뜀 없음)": O, "같은 수준 연번": O}
        notes, n_numbered, first_bad, skipped, sequence = {}, 0, [], [], []
        for name, items in segments:
            if not items:
                continue
            n_numbered += len(items)
            if items[0][1] != 1:
                first_bad.append(f'[{name}] p{items[0][0]} "{visible(items[0][3][:30])}" ({NAMES[items[0][1]]}) — 󰊱 로 시작해야 함')
            counters, current = {}, 0
            for i, level, n, text in items:
                if level > current + 1:
                    skipped.append(f'[{name}] p{i} "{visible(text[:30])}" — {NAMES[current] if current else "시작"} 다음에 {NAMES[level]} (중간 {", ".join(NAMES[x] for x in range(current + 1, level))} 건너뜀)')
                    current = level - 1
                expected = 1 if level > current else counters.get(level, 0) + 1  # 새 하위 목록이면 1부터
                if n != expected:
                    sequence.append(f'[{name}] p{i} "{visible(text[:30])}" — {NAMES[level]} {n}번째인데 {expected} 이어야 함')
                counters[level] = n
                for x in list(counters):
                    if x > level:
                        del counters[x]
                current = level
        if n_numbered == 0:
            checks = {"번호 찾기": TRI}
            notes["번호 찾기"] = "번호가 붙은 문단을 텍스트에서 찾지 못함 — 자동 번호(문단 번호 속성)일 수 있음, 원고 확인"
        else:
            if first_bad:
                checks["첫 수준이 󰊱"], notes["첫 수준이 󰊱"] = X, " / ".join(first_bad[:4])
            if skipped:
                checks["수준 순서(건너뜀 없음)"], notes["수준 순서(건너뜀 없음)"] = X, " / ".join(skipped[:6])
            if sequence:
                checks["같은 수준 연번"], notes["같은 수준 연번"] = X, " / ".join(sequence[:6])
            if table_paragraphs is None:
                notes["표 안 문단"] = "hwp 라 표 안 번호도 함께 셈(표 안 번호가 지적되면 무시)"
        rows.append(row("⑥", region, f"번호 문단 {n_numbered}개", checks, notes, "󰊱 〉 1. 〉 (1) 〉 (가) 〉 1) 〉 가) 〉 ① 〉 ㉠ 순서로 수정"))
    return rows
''')


In [ ]:
# 모듈: ncs_knowledge/rules/sources.py
write_module('ncs_knowledge/rules/sources.py', r'''"""F02 필요지식 ⑧: 그림·표 출처 표기 — 존재(그림 필수, 표 선택) · '출처:' 머리말 · F14 템플릿 골격 · 마침표 구분."""

import re

from ncs_core import NA, O, TRI, X, visible

from .common import row

SRC_HEAD = re.compile(r"^출\s*처\s*[:：]\s*")
YEAR = r"(?:\d{4}|\[\d{4}\]\s*\d{4}|\d{4}[a-z])"
DATE = r"\d{4}\.\s?\d{1,2}\.\s?\d{1,2}\."
TEMPLATES = [  # (이름, 정규식) — F14 '출처' 예시 14종의 골격
    ("집필진 제작", re.compile(rf"^집필진\s*제작\s*\({YEAR}\)\.?$")),
    ("전자 자료·검색", re.compile(rf"^.+?\(.+?\)에서\s*{DATE}\s*검색\.(\s*공공누리\s*제\d유형\.)?$")),
    ("스크린 숏", re.compile(rf"^.+?(\(.+?\)에서|\.)\s*{DATE}\s*스크린\s*숏\.$")),
    ("법령", re.compile(rf"^[^()]+\({DATE}(\s[^)]+)?\)\.?$")),
    ("문헌(저자(연도). 제목. 발행처. p.00.)", re.compile(rf"^[^()]{{1,80}}\({YEAR}\)\.\s.+\.\s*(pp?\.\s*\d+(-\d+)?\.?|.*\.)$")),
]
AI_MARK = re.compile(r"(AI|GPT|gpt|LM\b|Banana|Gemini|도식화|생성)")
OLD_AGENCY = re.compile(r"한국직업능력개발원")
CAPTION_START = re.compile(r"^[\[〈<]\s*(그림|표)")


def source_check(text: str) -> tuple[str, str]:
    """출처 줄 한 개 → (체크값, 비고 또는 템플릿 이름)."""
    if not SRC_HEAD.match(text):
        return X, '"출처:" 로 시작하지 않음'
    body = SRC_HEAD.sub("", text).strip()
    if not re.match(r"^출처:\s", text):
        return X, '"출처:" 뒤 띄어쓰기·콜론 모양이 양식과 다름(출처: ␣)'
    if AI_MARK.search(body) and not TEMPLATES[0][1].match(body):
        return TRI, f'AI 활용 표기 "{body}" — 지정 형식(집필진 제작(연도) 또는 직능연 AI 표기 양식)인지 확인'
    for name, pattern in TEMPLATES:
        if pattern.match(body):
            return O, name
    hints = []
    if re.search(r"\)\s*,", body):
        hints.append("연도 괄호 뒤가 쉼표 — 마침표로 구분")
    if not re.search(rf"\({YEAR}\)", body) and not re.search(DATE, body):
        hints.append("연도 괄호 (YYYY) 없음")
    if not body.endswith(".") and not TEMPLATES[0][1].match(body):
        hints.append("끝에 마침표 없음")
    if re.search(r"집필진\s*제작", body):
        hints.append("'집필진 제작(연도)' 꼴이 아님")
    return X, "F14 출처 템플릿과 골격이 다름" + (": " + ", ".join(hints) if hints else "")


def check_sources(body_raw: list[str], body_section: int, caps: list[dict], table_paragraphs: set[int] | None, *, lookback: int, lookahead: int) -> list[dict]:
    """caps: figtab.collect_labels 의 캡션 목록. 그림: 캡션 바로 위 '출처' 줄. 표: 표 아래 첫 '출처' 줄(선택)."""
    rows, used = [], set()
    in_table = lambda i: table_paragraphs is not None and i in table_paragraphs
    for cap in caps:
        if cap["sec"] != body_section:
            continue
        p, kind = cap["para"], cap["kind"]
        src_i = None
        if kind == "그림":
            for i in range(p - 1, max(p - 1 - lookback, -1), -1):
                if body_raw[i] and SRC_HEAD.match(body_raw[i]):
                    src_i = i
                    break
                if body_raw[i] and re.match(r"^\[그림", body_raw[i]):
                    break
        else:
            for i in range(p + 1, min(p + 1 + lookahead, len(body_raw))):
                if body_raw[i] and CAPTION_START.match(body_raw[i]):
                    break
                if body_raw[i] and SRC_HEAD.match(body_raw[i]) and not in_table(i):
                    src_i = i
                    break
        region = f"section1 > {cap['label']}"
        checks, notes = {}, {}
        if src_i is None:
            checks["출처 줄 존재"] = X if kind == "그림" else NA
            notes["출처 줄 존재"] = (
                '그림 바로 위에 출처 줄이 없음 — 집필진 제작도 "출처: 집필진 제작(연도)" 필수' if kind == "그림"
                else "표 아래 출처 없음 — 직접 작성한 표면 생략 가능(인용이면 필수)"
            )
            rows.append(row("⑧", region, cap["text"], checks, notes, "출처: 집필진 제작(연도) 또는 인용 문헌 형식으로 추가" if kind == "그림" else None))
            continue
        used.add(src_i)
        checks["출처 줄 존재"] = O
        if kind == "그림" and src_i != p - 1:
            notes["출처 줄 위치"] = f"출처 줄(p{src_i})과 캡션(p{p}) 사이에 다른 문단이 있음"
        value, note = source_check(body_raw[src_i])
        checks["출처 형식 골격"] = value
        if value == O:
            notes["템플릿"] = note
        else:
            notes["출처 형식 골격"] = note
        if OLD_AGENCY.search(body_raw[src_i]):
            checks["기관명"], notes["기관명"] = TRI, "'한국직업능력개발원' → 현재 명칭 '한국직업능력연구원'(F14) 인지 확인"
        rows.append(row("⑧", region, visible(body_raw[src_i]), checks, notes,
                        "출처: 집필진 제작(연도) / 출처: 저자(연도). 제목. 발행처. p.00. 꼴로 수정" if value == X else None, 캡션=cap["text"]))
    for i, line in enumerate(body_raw):  # 캡션과 짝이 없는 출처 줄
        if line and SRC_HEAD.match(line) and i not in used and not in_table(i):
            rows.append(row("⑧", f"section1 > p{i}", visible(line), {"캡션과 연결": TRI}, {"캡션과 연결": "이 출처 줄 앞뒤에 그림·표 캡션이 없음 — 캡션 누락 또는 떨어진 출처"}))
    return rows
''')


In [ ]:
# 모듈: ncs_knowledge/structure.py
write_module('ncs_knowledge/structure.py', r'''"""본문 구조 보조: 학습 내용(N-M) 구간·머리말, hwpx 표 셀 문단 집합."""

import re
import zipfile
from xml.etree import ElementTree as ET

from ncs_core.hwpx import HP, section_names

from .config import DEFAULT_CONFIG, CheckConfig

HEAD_MARK = {
    "필요 지식": re.compile(r"^필\s*요\s*지\s*식$"),
    "수행 내용": re.compile(r"^수\s*행\s*내\s*용$"),
    "수행 순서": re.compile(r"^수\s*행\s*순\s*서$"),
    "학습 목표": re.compile(r"^학\s*습\s*목\s*표$"),
    "재료": re.compile(r"^재\s*료(\s*\(.*\))?$"),
    "기기": re.compile(r"^기\s*기\s*\(장비\)$|^기기\s*\(장비\s*\)$|^기기$"),
    "안전": re.compile(r"^안\s*전(\s*\(.*\))?$"),
    "교수·학습 방법": re.compile(r"^교수\s*\W{0,3}\s*학습\s*방법$"),
    "평가": re.compile(r"^평\s*가$"),
}
MEMO_LINE = re.compile(r"^[★▶※]")           # 에디터 메모·주석 줄
GOAL_LINE = re.compile(r"^[•·▪ㆍ‧◦○\-]")   # 학습 목표 글머리


def mark(line: str) -> str | None:
    for name, pattern in HEAD_MARK.items():
        if pattern.match(line):
            return name
    return None


def unit_regions(body: dict, config: CheckConfig = DEFAULT_CONFIG) -> list[dict]:
    """학습 내용(N-M) 구간 목록.

    [{'no','sub','text','k0'(제목 lines 인덱스),'k1'(다음 제목/학습 끝),'lesson'(놓인 학습 번호),
      'goal':[학습 목표 문장], 'goal_k'('학습 목표' 줄 인덱스 또는 None)}]
    """
    lines, out = body["lines"], []
    subs = sorted(body["subs"], key=lambda s: s["k"])
    for j, sub in enumerate(subs):
        lesson = next((no for no, a, b in body["spans"] if a <= sub["k"] < b), None)
        k1 = subs[j + 1]["k"] if j + 1 < len(subs) else next((b for no, a, b in body["spans"] if a <= sub["k"] < b), len(lines))
        for k in range(sub["k"] + 1, k1):  # 교수·학습 방법 제목이 먼저 나오면 거기서 끊음(학습 끝)
            if mark(lines[k][1]) == "교수·학습 방법":
                k1 = k
                break
        goal, goal_k = [], None
        for k in range(sub["k"] + 1, min(sub["k"] + 1 + config.goal_lookahead, k1)):
            if mark(lines[k][1]) == "학습 목표":
                goal_k = k
                for kk in range(k + 1, k1):
                    text = lines[kk][1]
                    if mark(text) or MEMO_LINE.match(text):
                        break
                    if GOAL_LINE.match(text) or re.search(r"있다\.?$", text):
                        goal.append(text)
                    else:
                        break
                break
        out.append({"no": sub["no"], "sub": sub["sub"], "text": sub["text"], "k0": sub["k"], "k1": k1, "lesson": lesson, "goal": goal, "goal_k": goal_k})
    return out


def cell_paragraphs(path, section: int) -> set[int] | None:
    """hwpx: 해당 섹션에서 표 셀 안에 든 문단의 인덱스(ncs_core.hwpx와 같은 순회 순서). hwp면 None."""
    try:
        with zipfile.ZipFile(path) as archive:
            names = section_names(archive)
            if section >= len(names):
                return set()
            root = ET.fromstring(archive.read(names[section]))
    except zipfile.BadZipFile:
        return None
    parents = {child: parent for parent in root.iter() for child in parent}
    out = set()
    for index, paragraph in enumerate(root.iter(HP + "p")):
        current = parents.get(paragraph)
        while current is not None:
            if current.tag == HP + "tbl":
                out.add(index)
                break
            current = parents.get(current)
    return out
''')


In [ ]:
# 모듈: ncs_knowledge/text.py
write_module('ncs_knowledge/text.py', r'''"""필요 지식 검수의 문자 비교 규칙은 차례 검수와 같다 (머리 번호 '학습 1. ' / '1. ' / '1-1. ' 제거)."""

from ncs_toc.text import norm, same  # noqa: F401
''')


In [ ]:
# 모듈: ncs_overview/__init__.py
write_module('ncs_overview/__init__.py', r'''"""F02 학습모듈의 개요 ①·②·⑤·⑥·⑧ 검수. 읽기·판정·저장은 ncs_core를 쓴다."""

from ncs_core import ReviewResult, SavedReport

from .config import CheckConfig
from .pipeline import evaluate_review, prepare_review, run_review
from .registry import DEFAULT_RULES
from .reporting import save_results

__all__ = [
    "CheckConfig",
    "DEFAULT_RULES",
    "ReviewResult",
    "SavedReport",
    "prepare_review",
    "evaluate_review",
    "run_review",
    "save_results",
]
__version__ = "1.0.0"
''')


In [ ]:
# 모듈: ncs_overview/__main__.py
write_module('ncs_overview/__main__.py', r'''"""python -m ncs_overview 원고.hwpx [개발계획서.hwp] [--module-code LM...] [--out 결과]"""

import argparse
import json

from . import CheckConfig, run_review, save_results


def main() -> None:
    parser = argparse.ArgumentParser(description="F02 학습모듈의 개요 ①·②·⑤·⑥·⑧ 검수")
    parser.add_argument("paths", nargs="+", help="원고와 (선택) 개발계획서. 순서 무관, 내용으로 역할 판별")
    parser.add_argument("--manuscript", help="자동 역할 판별 대신 원고로 지정할 입력 경로")
    parser.add_argument("--plan", help="자동 역할 판별 대신 개발계획서로 지정할 입력 경로")
    parser.add_argument("--module-code", default="", help="비우면 원고 파일명에서 추출")
    parser.add_argument("--front-section", type=int, default=0)
    parser.add_argument("--out", default="결과")
    args = parser.parse_args()
    roles = {}
    if args.manuscript:
        roles[args.manuscript] = "manuscript"
    if args.plan:
        roles[args.plan] = "plan"
    try:
        config = CheckConfig(front_section=args.front_section)
        result = run_review(args.paths, roles=roles, module_code=args.module_code, config=config)
        saved = save_results(result, args.out)
    except (OSError, ValueError) as error:
        parser.exit(2, f"검수 실행 실패: {error}\n")
    print(json.dumps({"요약": saved.summary, "진단": result.diagnostics}, ensure_ascii=False, indent=2))
    print(f"지적사항 {len(result.findings)}건 · {saved.zip_path}")


if __name__ == "__main__":
    main()
''')


In [ ]:
# 모듈: ncs_overview/compare.py
write_module('ncs_overview/compare.py', r'''"""목록 비교 도우미. ncs_toc.compare.diff_list와 같은 정책(difflib, 누락/추가/변경/순서 구분)을 개요 정규화 규칙으로 쓴다."""

import difflib
from collections.abc import Sequence

from ncs_core import O, X, visible

from .text import norm


def diff_list(label: str, expected: Sequence[str], actual: Sequence[str]) -> tuple[str, list[str]]:
    exp, act = [norm(x) for x in expected], [norm(x) for x in actual]
    notes: list[str] = []
    if len(expected) != len(actual):
        notes.append(f"{label} 개수: 원고 {len(actual)} / 개발계획서 {len(expected)}")
    if exp != act and sorted(exp) == sorted(act):
        notes.append(f"{label} 순서 다름: 원고 {[visible(x) for x in actual]} / 개발계획서 {list(expected)}")
        return X, notes
    matcher = difflib.SequenceMatcher(None, exp, act, autojunk=False)
    for op, i1, i2, j1, j2 in matcher.get_opcodes():
        if op == "equal":
            continue
        if op == "replace" and i2 - i1 == j2 - j1:
            for k in range(i2 - i1):
                notes.append(f'{label} {i1 + k + 1}번째 다름: "{visible(actual[j1 + k])}" / 개발계획서 "{expected[i1 + k]}"')
        else:
            for k in range(i1, i2):
                notes.append(f'{label} {k + 1}번째 누락: "{expected[k]}"')
            for k in range(j1, j2):
                notes.append(f'{label} 원고 {k + 1}번째가 개발계획서에 없음: "{visible(actual[k])}"')
    return (O if not notes else X), notes
''')


In [ ]:
# 모듈: ncs_overview/config.py
write_module('ncs_overview/config.py', r'''"""개요 검수의 조정 가능한 양식 가정."""
from dataclasses import dataclass


@dataclass(frozen=True)
class CheckConfig:
    front_section: int = 0          # 학습모듈의 개요가 있는 섹션
    max_prerequisites: int = 5      # ② 선수학습 최대 개수
    max_terms: int = 10             # ⑧ 핵심 용어 최대 개수
    term_max_length: int = 25       # ⑧ 이보다 긴 항목은 단어가 아닐 가능성 → △

    def __post_init__(self):
        if self.front_section < 0:
            raise ValueError("섹션 번호는 0 이상이어야 합니다")
        for name in ("max_prerequisites", "max_terms", "term_max_length"):
            if getattr(self, name) < 1:
                raise ValueError(f"{name}은 1 이상이어야 합니다")


DEFAULT_CONFIG = CheckConfig()
''')


In [ ]:
# 모듈: ncs_overview/models.py
write_module('ncs_overview/models.py', r'''"""파서와 규칙 사이의 경계. 결과 모델(ReviewResult 등)은 ncs_core.models를 쓴다."""

from dataclasses import dataclass
from typing import Any

from ncs_core import Document

from .config import CheckConfig


@dataclass(frozen=True)
class ReviewContext:
    manuscript: Document
    plan: Document | None
    module_code: str
    config: CheckConfig
    overview: dict[str, Any]          # parsers.parse_overview 결과
    plan_info: dict[str, Any] | None  # parsers.parse_plan 결과 (계획서 없으면 None)
''')


In [ ]:
# 모듈: ncs_overview/parsers.py
write_module('ncs_overview/parsers.py', r'''"""학습모듈의 개요(목표·선수학습·내용체계·핵심 용어)와 개발계획서 모듈 블록을 추출한다.

내용체계는 표 셀(hwp·hwpx 모두 표 셀 텍스트가 있다)을 우선 읽고, 표를 못 찾으면 문단 줄로 읽는다.
차례 검수(ncs_toc.parsers)의 OV_LESSON/OV_SUB 규칙과 같은 정규식을 쓴다.
"""

import re

from ncs_core import Document, Table

from .patterns import (
    ELEMENT_CODE,
    HEAD_GOAL,
    HEAD_OVERVIEW,
    HEAD_PREREQ,
    HEAD_STRUCTURE,
    HEAD_TERMS,
    OV_LESSON,
    OV_SUB,
    PLAN_LESSON,
)
from .text import split_items

HEADINGS = (HEAD_GOAL, HEAD_PREREQ, HEAD_STRUCTURE, HEAD_TERMS)
STRUCTURE_HEADERS = ("학습", "학습 내용", "코드번호", "요소 명칭")


def _block(lines: list[tuple[int, str]], head, stop_heads) -> tuple[int | None, list[tuple[int, str]]]:
    """head 제목 다음부터 다음 제목(stop_heads) 전까지의 (문단 인덱스, 텍스트)."""
    start = next((k for k, (_, l) in enumerate(lines) if head.match(l)), None)
    if start is None:
        return None, []
    out = []
    for i, l in lines[start + 1 :]:
        if any(h.match(l) for h in stop_heads):
            break
        out.append((i, l))
    return lines[start][0], out


def _structure_from_table(table: Table) -> dict | None:
    cells = table["cells"]
    headers: dict[str, tuple[int, int]] = {}
    for (r, c), v in sorted(cells.items()):
        key = v.replace(" ", "")
        for name in STRUCTURE_HEADERS:
            if key == name.replace(" ", "") and name not in headers:
                headers[name] = (r, c)
    if not {"학습", "코드번호"} <= headers.keys():
        return None
    header_row = max(r for r, _ in headers.values())
    col = {name: c for name, (_, c) in headers.items()}
    content_col = col.get("학습 내용", col["학습"] + 1)
    name_col = col.get("요소 명칭", col["코드번호"] + 1)
    lessons, contents, elements = [], [], []
    rows = sorted({r for r, _ in cells if r > header_row})
    for r in rows:
        lesson, content = cells.get((r, col["학습"]), ""), cells.get((r, content_col), "")
        code, name = cells.get((r, col["코드번호"]), ""), cells.get((r, name_col), "")
        if m := OV_LESSON.match(lesson):
            lessons.append({"no": int(m.group(1)), "text": m.group(2).strip(), "row": r})
        if m := OV_SUB.match(content):
            contents.append({"no": int(m.group(1)), "sub": int(m.group(2)), "text": m.group(3).strip(), "row": r})
        for part in code.split():
            if ELEMENT_CODE.match(part) and not any(e["code"] == part for e in elements):
                elements.append({"code": part, "name": name, "row": r})
    return {"학습": lessons, "내용": contents, "요소": elements, "근거": f"표 (p{table['paragraph']})"}


def _structure_from_lines(block: list[tuple[int, str]]) -> dict:
    lessons, contents, elements = [], [], []
    for k, (i, l) in enumerate(block):
        if (m := OV_LESSON.match(l)) and int(m.group(1)) == len(lessons) + 1:
            lessons.append({"no": int(m.group(1)), "text": m.group(2).strip(), "row": i})
        elif m := OV_SUB.match(l):
            text = m.group(3).strip() or (block[k + 1][1] if k + 1 < len(block) else "")
            contents.append({"no": int(m.group(1)), "sub": int(m.group(2)), "text": text, "row": i})
        elif m := ELEMENT_CODE.match(l):
            # 요소 명칭은 다음 줄(셀 안에서 줄이 나뉘면 두 줄)
            name_parts = []
            for _, nxt in block[k + 1 : k + 3]:
                if OV_LESSON.match(nxt) or OV_SUB.match(nxt) or ELEMENT_CODE.match(nxt):
                    break
                name_parts.append(nxt)
            elements.append({"code": l, "name": " ".join(name_parts).strip(), "row": i})
    return {"학습": lessons, "내용": contents, "요소": elements, "근거": "문단 줄"}


def parse_overview(document: Document, section: int) -> dict:
    """→ {'목표': {'text','p'}|None, '선수학습': {'items','text','p'}|None,
         '내용체계': {...}|None, '핵심 용어': {'items','text','p'}|None, '개요 제목 p': int|None}"""
    raw = document.sections[section] if section < len(document.sections) else []
    lines = [(i, l) for i, l in enumerate(raw) if l]
    head = next((i for i, l in lines if HEAD_OVERVIEW.search(l) and not re.search(r"\d+$", l)), None)
    if head is not None:
        lines = [(i, l) for i, l in lines if i > head]

    out: dict = {"개요 제목 p": head}
    p, block = _block(lines, HEAD_GOAL, HEADINGS)
    out["목표"] = {"text": " ".join(l for _, l in block), "p": block[0][0]} if block else None
    p, block = _block(lines, HEAD_PREREQ, HEADINGS)
    if block:
        text = " ".join(l for _, l in block)
        out["선수학습"] = {"items": split_items(text), "text": text, "p": block[0][0]}
    else:
        out["선수학습"] = None
    p, block = _block(lines, HEAD_STRUCTURE, HEADINGS)
    structure = None
    if p is not None:
        tables = [t for t in document.section_tables(section) if t["paragraph"] >= p]
        structure = next((s for t in tables if (s := _structure_from_table(t))), None)
        if structure is None and block:
            structure = _structure_from_lines(block)
    out["내용체계"] = structure
    p, block = _block(lines, HEAD_TERMS, HEADINGS)
    if block:
        text = " ".join(l for _, l in block)
        out["핵심 용어"] = {"items": split_items(text), "text": text, "p": block[0][0]}
    else:
        out["핵심 용어"] = None
    return out


def parse_plan(document: Document, module_code: str) -> dict | None:
    """개발계획서 모듈 블록 → {'정의','목표','학습명':[...],'내용명':[{no,sub,text}],'요소':[{code,name}]}.

    블록은 '능력단위 코드번호' 다음 줄의 코드부터 'NCS (수정 금지)' 전까지 (ncs_toc.parsers.parse_plan_toc와 같은 기준).
    """
    lines = [l for section in document.sections for l in section]
    code = module_code.replace("LM", "")
    start = next(
        (i for i, l in enumerate(lines) if l == code and i > 0 and lines[i - 1].startswith("능력단위 코드번호")),
        None,
    )
    if start is None:
        return None
    end = next((i for i in range(start, len(lines)) if lines[i].startswith("NCS (수정 금지)")), len(lines))
    block = [l for l in lines[start:end] if l]
    plan = {"정의": None, "목표": None, "학습명": [], "내용명": [], "요소": []}
    for i, line in enumerate(block):
        nxt = block[i + 1] if i + 1 < len(block) else ""
        if re.fullmatch(r"능력단위\s*정의", line) and plan["정의"] is None:
            plan["정의"] = nxt
        elif re.fullmatch(r"학습모듈의\s*목표", line) and plan["목표"] is None:
            plan["목표"] = nxt
        elif (m := ELEMENT_CODE.match(line)) and m.group(1) == code:
            plan["요소"].append({"code": line, "name": nxt})
        elif m := OV_SUB.match(line):
            text = m.group(3).strip() or nxt
            plan["내용명"].append({"no": int(m.group(1)), "sub": int(m.group(2)), "text": text})
        elif (m := PLAN_LESSON.match(line)) and int(m.group(1)) == len(plan["학습명"]) + 1:
            plan["학습명"].append(m.group(2).strip())
    return plan
''')


In [ ]:
# 모듈: ncs_overview/patterns.py
write_module('ncs_overview/patterns.py', r'''"""개요 양식의 제목·내용 인식 패턴. 판정 정책은 rules/에 둔다."""
import re

HEAD_OVERVIEW = re.compile(r"학습모듈의\s*개요$")
HEAD_GOAL = re.compile(r"^학습모듈의\s*목표$")
HEAD_PREREQ = re.compile(r"^선수\s*학습$")
HEAD_STRUCTURE = re.compile(r"^학습모듈의\s*내용\s*체계$")
HEAD_TERMS = re.compile(r"^핵심\s*용어$")
OV_LESSON = re.compile(r"^(\d+)\.\s*(\S.*)$")           # 1. 명 (개요 표)
PLAN_LESSON = re.compile(r"^(\d+)\.\s+(\S.*)$")         # 계획서 학습명: 공백 필수 ("1.1 …" 수행준거와 구별)
OV_SUB = re.compile(r"^(\d+)-(\d+)\.\s*(.*)$")          # 1-1. 명 (명이 다음 줄에 올 수도 있음)
ELEMENT_CODE = re.compile(r"^(\d{10}_\d+v\d+)\.(\d+)$")  # 1203020201_24v3.1
GOAL_END = re.compile(r"할\s*수\s*있다\s*\.$")
# 능력단위 정의 → 학습모듈의 목표: 주어('…란/이란/은/는') 삭제, '~하는 능력이다' → '~할 수 있다'
DEFINITION_SUBJECT = re.compile(r"^.*?(?:이란|란|은|는)\s+")
DEFINITION_END = re.compile(r"(하|할 수 있)는\s*능력이다\s*\.?$")
# ⑧ 단어(명사형)가 아닐 가능성이 있는 종결
TERM_PREDICATE = re.compile(r"(다|하기|함|됨|하는|되는|할 것|것)$")
''')


In [ ]:
# 모듈: ncs_overview/pipeline.py
write_module('ncs_overview/pipeline.py', r'''"""입력 선택 → 추출 → 규칙 실행. 저장과 화면 출력은 호출자가 결정한다. (ncs_toc.pipeline과 같은 구조)"""

import re
from collections.abc import Sequence
from typing import Any

from ncs_core import Document, ReviewResult, Rule, resolve_module_code, run_rules, select_documents
from ncs_core.text import flatten

from .config import DEFAULT_CONFIG, CheckConfig
from .models import ReviewContext
from .parsers import parse_overview, parse_plan
from .registry import ALL_ITEM_IDS, DEFAULT_RULES


def prepare_review(
    manuscript: Document,
    plan: Document | None = None,
    *,
    module_code: str = "",
    config: CheckConfig = DEFAULT_CONFIG,
) -> ReviewContext:
    code = resolve_module_code(manuscript.path, module_code)
    if config.front_section >= len(manuscript.sections):
        raise ValueError("지정한 개요 섹션이 없습니다. 문서 구조와 CheckConfig를 확인하세요")
    overview = parse_overview(manuscript, config.front_section)
    if overview["개요 제목 p"] is None and not any(overview.get(k) for k in ("목표", "선수학습", "내용체계", "핵심 용어")):
        raise ValueError("학습모듈의 개요를 읽지 못했습니다. 양식과 섹션 설정을 확인하세요")
    plan_info = parse_plan(plan, code) if plan else None
    return ReviewContext(manuscript, plan, code, config, overview, plan_info)


def _actual_sections(value: Any, config: CheckConfig) -> Any:
    if isinstance(value, str):
        return re.sub(r"section0\b", f"section{config.front_section}", value)
    if isinstance(value, dict):
        return {k: _actual_sections(v, config) for k, v in value.items()}
    if isinstance(value, list):
        return [_actual_sections(v, config) for v in value]
    return value


def _diagnostics(context: ReviewContext, rules: Sequence[Rule]) -> dict:
    manuscript, plan, overview = context.manuscript, context.plan, context.overview
    warnings = []
    if plan and not context.plan_info:
        warnings.append("개발계획서에서 이 모듈 코드의 블록을 찾지 못했습니다. ①⑤⑥ 대조는 생략·판단불가입니다")
    elif not plan:
        warnings.append("개발계획서가 없어 ①⑤⑥의 개발계획서 대조는 생략·판단불가입니다")
    elif not context.plan_info["요소"] or not context.plan_info["학습명"]:
        warnings.append("개발계획서 요소 코드 또는 학습명을 읽지 못했습니다")
    for key in ("목표", "선수학습", "내용체계", "핵심 용어"):
        if not overview.get(key):
            warnings.append(f"개요의 '{key}' 블록을 읽지 못했습니다")
    implemented = {rule.item_prefix for rule in rules}
    structure = overview.get("내용체계") or {}
    return {
        "module_code": context.module_code,
        "inputs": {"manuscript": str(manuscript.path), "plan": str(plan.path) if plan else None},
        "format": manuscript.format,
        "sections": {"front": context.config.front_section},
        "counts": {
            "prerequisites": len(overview["선수학습"]["items"]) if overview.get("선수학습") else 0,
            "lessons": len(structure.get("학습", [])),
            "contents": len(structure.get("내용", [])),
            "elements": len(structure.get("요소", [])),
            "terms": len(overview["핵심 용어"]["items"]) if overview.get("핵심 용어") else 0,
        },
        "implemented": sorted(implemented),
        "not_implemented": [item for item in ALL_ITEM_IDS if item not in implemented],
        "warnings": warnings,
    }


def evaluate_review(context: ReviewContext, *, rules: Sequence[Rule] = DEFAULT_RULES) -> ReviewResult:
    config = context.config
    checklist = run_rules(context, rules)
    for row in checklist:
        row["region_path"] = _actual_sections(row["region_path"], config)
        row["비고"] = _actual_sections(row["비고"], config)
    diagnostics = _diagnostics(context, rules)
    return ReviewResult(context.module_code, checklist, flatten(checklist), diagnostics, context)


def run_review(
    paths,
    *,
    roles: dict | None = None,
    module_code: str = "",
    config: CheckConfig = DEFAULT_CONFIG,
    rules: Sequence[Rule] = DEFAULT_RULES,
) -> ReviewResult:
    documents = select_documents(paths, roles, required=("manuscript",))
    manuscript, plan = documents["manuscript"], documents.get("plan")
    context = prepare_review(manuscript, plan, module_code=module_code, config=config)
    return evaluate_review(context, rules=rules)
''')


In [ ]:
# 모듈: ncs_overview/registry.py
write_module('ncs_overview/registry.py', r'''"""규칙 추가 지점. 규칙은 ReviewContext 하나를 받아 기존 형식의 행 목록을 돌려준다."""

from ncs_core import Rule

from .rules.goal import check_goal
from .rules.prerequisites import check_prerequisites
from .rules.structure import check_structure_elements, check_structure_names
from .rules.terms import check_terms

DEFAULT_RULES = (
    Rule("goal", "F02-개요-①", lambda c: check_goal(c.overview, c.plan_info)),
    Rule("prerequisites", "F02-개요-②", lambda c: check_prerequisites(c.overview, c.config.max_prerequisites)),
    Rule("structure_names", "F02-개요-⑤", lambda c: check_structure_names(c.overview, c.plan_info)),
    Rule("structure_elements", "F02-개요-⑥", lambda c: check_structure_elements(c.overview, c.plan_info, c.module_code)),
    Rule("terms", "F02-개요-⑧", lambda c: check_terms(c.overview, c.config.max_terms, c.config.term_max_length)),
)

# 개요 항목 ①~⑧ 중 규칙이 없는 것은 진단의 not_implemented로 보고한다.
# ③ 선수학습 코드 병기, ④ 선수학습 능력단위 수준(NCS 수준 자료 필요), ⑦ 통합 개발 시 요소 전부 기재(통합 사례 필요).
ALL_ITEM_IDS = tuple("F02-개요-" + n for n in "①②③④⑤⑥⑦⑧")
''')


In [ ]:
# 모듈: ncs_overview/reporting.py
write_module('ncs_overview/reporting.py', r'''"""개요 검수 결과 저장. 파일 이름 규칙은 ncs_core.reporting과 같다."""

from ncs_core import ReviewResult, SavedReport, save_report

ITEM_FILES = (
    ("F02-개요-①", "개요①_목표"),
    ("F02-개요-②", "개요②_선수학습"),
    ("F02-개요-⑤", "개요⑤_학습명내용명"),
    ("F02-개요-⑥", "개요⑥_요소코드명칭"),
    ("F02-개요-⑧", "개요⑧_핵심용어"),
)


def save_results(result: ReviewResult, out_dir="결과", *, per_run_dir: bool = True) -> SavedReport:
    return save_report(
        result, out_dir, name="개요", item_files=ITEM_FILES, other_tail="개요_기타", per_run_dir=per_run_dir
    )
''')


In [ ]:
# 모듈: ncs_overview/rules/__init__.py
write_module('ncs_overview/rules/__init__.py', r'''"""문서 읽기와 독립적인 개요 검수 규칙."""
''')


In [ ]:
# 모듈: ncs_overview/rules/goal.py
write_module('ncs_overview/rules/goal.py', r'''"""F02 개요 ①: '학습모듈의 목표' — NCS 능력단위 정의에서 주어를 삭제한 후 '~할 수 있다.'로 종결하였는가.

형식 항목이다. 계획서 문장과 글자가 같은지를 묻지 않으므로
  - 종결어와 주어 삭제는 O/X로 판정하고,
  - 정의와의 관계는 내용이 대응하는지(공백·구두점을 뺀 유사도)만 보아 낮으면 △(사람 확인)로 둔다.
계획서의 '학습모듈의 목표' 줄은 판정에 쓰지 않고 참고로만 적는다.
"""

import difflib
import re

from ncs_core import NA, O, PASS, TRI, X, verdict

from ..patterns import DEFINITION_END, DEFINITION_SUBJECT, GOAL_END

ITEM_ID = "F02-개요-①"
SIMILARITY_MIN = 0.85  # 정의(주어 제거·서술어 변경 후)와 목표의 글자 유사도 하한. 이보다 낮으면 △
SUBJECT_HEAD = re.compile(r"^\S+(이란|란)\s")  # '객실 예약 접수란 ' 꼴 주어가 목표 머리에 남아 있음


def goal_from_definition(definition: str) -> tuple[str, str]:
    """'객실 예약 접수란 … 처리하는 능력이다.' → (주어 '객실 예약 접수란', '… 처리할 수 있다.')"""
    definition = definition.strip()
    m = DEFINITION_SUBJECT.match(definition)
    subject = m.group(0).strip() if m else ""
    body = definition[m.end():] if m else definition
    return subject, DEFINITION_END.sub("할 수 있다.", body).strip()


def _compact(text: str) -> str:
    """공백·구두점·기호 글꼴 가운뎃점을 뺀 글자열 (내용 대응 비교용)."""
    return re.sub(r"[\s·,.\-]", "", text)


def check_goal(overview: dict, plan: dict | None) -> list[dict]:
    goal = overview.get("목표")
    label_end = "① '~할 수 있다.' 종결"
    label_subject = "① 주어 삭제"
    label_def = "① 능력단위 정의 내용과 대응"
    notes: dict[str, str] = {}
    if not goal:
        checks = {label_end: X, label_subject: NA, label_def: NA}
        notes[label_end] = "'학습모듈의 목표' 제목 또는 내용을 찾지 못함"
        return [_row(None, None, checks, notes, "학습모듈의 목표를 작성할 것")]

    text = goal["text"].strip()
    definition = plan.get("정의") if plan else None
    subject, derived = goal_from_definition(definition) if definition else ("", "")

    checks = {label_end: O if GOAL_END.search(text) else X}
    if checks[label_end] == X:
        notes[label_end] = "'~할 수 있다.'로 종결되지 않음"

    # 주어 삭제: 정의의 주어가 목표 머리에 남아 있거나, '…란/이란' 꼴로 시작하면 X
    head = _compact(text)
    if subject and head.startswith(_compact(subject)):
        checks[label_subject] = X
        notes[label_subject] = f'정의의 주어 "{subject}" 가 남아 있음'
    elif SUBJECT_HEAD.match(text):
        checks[label_subject] = X
        notes[label_subject] = f'주어로 보이는 머리 "{SUBJECT_HEAD.match(text).group(0).strip()}" 가 남아 있음'
    else:
        checks[label_subject] = O

    # 정의와의 관계: 글자 일치가 아니라 내용 대응. 낮으면 △
    if definition:
        ratio = difflib.SequenceMatcher(None, _compact(derived), head, autojunk=False).ratio()
        if ratio >= SIMILARITY_MIN:
            checks[label_def] = O
            if _compact(derived) != head:
                notes[label_def] = f'정의와 표현이 조금 다름(유사도 {ratio:.2f}) — 정의 기준 문장: "{derived}"'
        else:
            checks[label_def] = TRI
            notes[label_def] = f'정의와 내용이 다름(유사도 {ratio:.2f}) — 사람 확인. 정의 기준 문장: "{derived}"'
    else:
        checks[label_def] = NA
        notes[label_def] = "개발계획서를 읽지 못해 대조 생략" if not plan else "개발계획서에 능력단위 정의가 없어 대조 생략"

    reference = {"능력단위 정의": definition, "개발계획서 목표(참고)": plan.get("목표")} if plan else None
    result = verdict(checks)
    if result == PASS:
        suggestion = None
    elif checks[label_end] == X or checks[label_subject] == X:
        parts = []
        if checks[label_subject] == X:
            parts.append("주어를 지우고")
        if checks[label_end] == X:
            parts.append("'~할 수 있다.'로 끝맺을 것")
        suggestion = "능력단위 정의에서 " + " ".join(parts)
    else:
        suggestion = "능력단위 정의의 내용과 대응하는지 확인"
    return [_row({"목표": text, "p": goal["p"]}, reference, checks, notes, suggestion)]


def _row(evidence, reference, checks, notes, suggestion) -> dict:
    return {
        "item_id": ITEM_ID,
        "region_path": "section0 > 학습모듈의 개요 > 학습모듈의 목표",
        "원고": evidence,
        "개발계획서": reference,
        "체크": checks,
        "판정": verdict(checks),
        "비고": notes,
        "수정 제안": suggestion,
    }
''')


In [ ]:
# 모듈: ncs_overview/rules/prerequisites.py
write_module('ncs_overview/rules/prerequisites.py', r'''"""F02 개요 ②: '선수학습' 항목 수가 최대 개수(기본 5) 이내인가. '구체적인'은 사람 판단."""

from ncs_core import O, PASS, X, verdict

ITEM_ID = "F02-개요-②"


def check_prerequisites(overview: dict, max_items: int) -> list[dict]:
    block = overview.get("선수학습")
    label = f"② 선수학습 {max_items}개 이내"
    notes: dict[str, str] = {}
    if not block or not block["items"]:
        checks = {label: X}
        notes[label] = "'선수학습' 제목 또는 내용을 찾지 못함"
        evidence = None
    else:
        n = len(block["items"])
        checks = {label: O if n <= max_items else X}
        if n > max_items:
            notes[label] = f"선수학습 {n}개 — {max_items}개 이내로 줄일 것"
        evidence = {"항목": block["items"], "개수": n, "p": block["p"]}
    result = verdict(checks)
    return [
        {
            "item_id": ITEM_ID,
            "region_path": "section0 > 학습모듈의 개요 > 선수학습",
            "원고": evidence,
            "체크": checks,
            "판정": result,
            "비고": notes,
            "수정 제안": None if result == PASS else f"구체적인 과목명·학습모듈명(또는 NCS 능력단위)을 {max_items}개 이내로 제시",
        }
    ]
''')


In [ ]:
# 모듈: ncs_overview/rules/structure.py
write_module('ncs_overview/rules/structure.py', r'''"""F02 개요 ⑤·⑥: '학습모듈의 내용 체계' 표의 학습명·학습 내용명(⑤), NCS 능력단위 요소 코드·명칭(⑥)이 개발계획서와 일치하는가."""

from ncs_core import NA, O, PASS, PUA, TRI, X, verdict, visible

from ..compare import diff_list
from ..text import same

ITEM_ID_5 = "F02-개요-⑤"
ITEM_ID_6 = "F02-개요-⑥"
REGION = "section0 > 학습모듈의 개요 > 학습모듈의 내용 체계"


def _missing(item_id: str, label: str, note: str) -> dict:
    return {
        "item_id": item_id,
        "region_path": REGION,
        "원고": None,
        "체크": {label: X},
        "판정": verdict({label: X}),
        "비고": {label: note},
        "수정 제안": "학습모듈의 내용 체계 표를 작성할 것",
    }


def check_structure_names(overview: dict, plan: dict | None) -> list[dict]:
    """⑤ 학습명·학습 내용명 ↔ 개발계획서. 차례 면과의 일치는 차례 검수 ③ 결과를 참조한다."""
    structure = overview.get("내용체계")
    label_l, label_c = "⑤ 학습명 개발계획서 일치", "⑤ 학습 내용명 개발계획서 일치"
    if not structure or not structure["학습"]:
        return [_missing(ITEM_ID_5, label_l, "'학습모듈의 내용 체계' 표 또는 학습명을 읽지 못함")]
    lessons = [visible(f"{L['no']}. {L['text']}") for L in structure["학습"]]
    contents = [visible(f"{s['no']}-{s['sub']}. {s['text']}") for s in structure["내용"]]
    notes: dict = {"근거": structure["근거"]}
    pua = [visible(t) for t in [L["text"] for L in structure["학습"]] + [s["text"] for s in structure["내용"]] if PUA.search(t)]
    if pua:
        # 비교는 기호 글꼴 문자를 가운뎃점으로 치환해서 하고, 문자 자체는 차례 ①에서 X로 보고한다.
        notes["기호 글꼴 문자(참고)"] = "가운뎃점이 기호 글꼴 문자 — ·(U+00B7)로 교체 (차례 ①에서 보고): " + "; ".join(pua)
    if not plan:
        checks = {label_l: TRI, label_c: TRI}
        notes[label_l] = notes[label_c] = "개발계획서를 읽지 못해 대조 불가 — 사람 확인"
    else:
        exp_l = plan["학습명"]
        exp_c = [f"{s['no']}-{s['sub']}. {s['text']}" for s in plan["내용명"]]
        if exp_l:
            value, diff = diff_list("학습명", exp_l, [L["text"] for L in structure["학습"]])
            checks = {label_l: value}
            if diff:
                notes[label_l] = "; ".join(diff)
        else:
            checks = {label_l: TRI}
            notes[label_l] = "개발계획서 학습명을 읽지 못함"
        if exp_c:
            value, diff = diff_list("학습 내용명", exp_c, contents)
            checks[label_c] = value
            if diff:
                notes[label_c] = "; ".join(diff)
        else:
            checks[label_c] = TRI
            notes[label_c] = "개발계획서 학습 내용명을 읽지 못함"
    result = verdict(checks)
    return [
        {
            "item_id": ITEM_ID_5,
            "region_path": REGION,
            "원고": {"학습": lessons, "학습 내용": contents},
            "개발계획서": {"학습명": plan["학습명"], "내용명": [f"{s['no']}-{s['sub']}. {s['text']}" for s in plan["내용명"]]} if plan else None,
            "체크": checks,
            "판정": result,
            "비고": notes,
            "수정 제안": None if result == PASS else "개발계획서의 학습명·학습 내용명과 같게 수정 (차례 면도 함께 확인)",
        }
    ]


def check_structure_elements(overview: dict, plan: dict | None, module_code: str) -> list[dict]:
    """⑥ NCS 능력단위 요소 코드 번호·요소 명칭 ↔ 개발계획서."""
    structure = overview.get("내용체계")
    label_set, label_name, label_prefix = "⑥ 요소 코드 번호 개발계획서 일치", "⑥ 요소 명칭 개발계획서 일치", "⑥ 코드 번호가 이 모듈의 능력단위 코드"
    if not structure or not structure["요소"]:
        return [_missing(ITEM_ID_6, label_set, "'학습모듈의 내용 체계' 표에서 요소 코드 번호를 읽지 못함")]
    elements = structure["요소"]
    unit = module_code.replace("LM", "")
    notes: dict = {"근거": structure["근거"]}
    wrong_prefix = [e["code"] for e in elements if not e["code"].startswith(unit + ".")]
    checks = {label_prefix: X if wrong_prefix else O}
    if wrong_prefix:
        notes[label_prefix] = f"능력단위 코드 {unit} 가 아닌 요소 코드: {wrong_prefix}"
    if not plan or not plan["요소"]:
        checks[label_set] = checks[label_name] = TRI
        notes[label_set] = "개발계획서 요소 코드를 읽지 못해 대조 불가 — 사람 확인" if plan else "개발계획서를 읽지 못해 대조 불가 — 사람 확인"
    else:
        exp = {e["code"]: e["name"] for e in plan["요소"]}
        got = {e["code"]: e["name"] for e in elements}
        missing, extra = sorted(exp.keys() - got.keys()), sorted(got.keys() - exp.keys())
        checks[label_set] = O if not missing and not extra else X
        parts = []
        if missing:
            parts.append(f"표에 없는 개발계획서 요소 코드: {missing}")
        if extra:
            parts.append(f"개발계획서에 없는 코드: {extra}")
        if parts:
            notes[label_set] = "; ".join(parts)
        bad = [f'{code}: "{got[code]}" / 기준 "{exp[code]}"' for code in got if code in exp and not same(exp[code], got[code])]
        checks[label_name] = X if bad else (O if got.keys() & exp.keys() else NA)
        if bad:
            notes[label_name] = "요소 명칭 다름 — " + "; ".join(bad)
    result = verdict(checks)
    return [
        {
            "item_id": ITEM_ID_6,
            "region_path": REGION,
            "원고": [{"코드번호": e["code"], "요소 명칭": visible(e["name"])} for e in elements],
            "개발계획서": plan["요소"] if plan else None,
            "체크": checks,
            "판정": result,
            "비고": notes,
            "수정 제안": None if result == PASS else "개발계획서의 능력단위 요소 코드 번호·요소명과 같게 수정",
        }
    ]
''')


In [ ]:
# 모듈: ncs_overview/rules/terms.py
write_module('ncs_overview/rules/terms.py', r'''"""F02 개요 ⑧: '핵심 용어'가 최대 개수(기본 10) 이내의 단어(명사형)인가. 명사형 여부는 휴리스틱으로 △만 낸다."""

from ncs_core import O, PASS, TRI, X, verdict

from ..patterns import TERM_PREDICATE

ITEM_ID = "F02-개요-⑧"


def check_terms(overview: dict, max_items: int, max_length: int) -> list[dict]:
    block = overview.get("핵심 용어")
    label_n, label_form = f"⑧ 핵심 용어 {max_items}개 이내", "⑧ 단어(명사형) 형태"
    notes: dict[str, str] = {}
    if not block or not block["items"]:
        checks = {label_n: X}
        notes[label_n] = "'핵심 용어' 제목 또는 내용을 찾지 못함"
        evidence = None
    else:
        items = block["items"]
        n = len(items)
        checks = {label_n: O if n <= max_items else X}
        if n > max_items:
            notes[label_n] = f"핵심 용어 {n}개 — {max_items}개 이내로 줄일 것"
        suspicious = [t for t in items if TERM_PREDICATE.search(t) or len(t) > max_length or t.endswith(".")]
        checks[label_form] = TRI if suspicious else O
        if suspicious:
            notes[label_form] = "단어(명사형)가 아닐 수 있는 항목 — 사람 확인: " + "; ".join(f'"{t}"' for t in suspicious)
        evidence = {"항목": items, "개수": n, "p": block["p"]}
    result = verdict(checks)
    return [
        {
            "item_id": ITEM_ID,
            "region_path": "section0 > 학습모듈의 개요 > 핵심 용어",
            "원고": evidence,
            "체크": checks,
            "판정": result,
            "비고": notes,
            "수정 제안": None if result == PASS else f"핵심 용어를 {max_items}개 이내의 단어(명사형)로 제시",
        }
    ]
''')


In [ ]:
# 모듈: ncs_overview/text.py
write_module('ncs_overview/text.py', r'''"""개요 검수의 문자 비교 규칙. 머리 번호 '1. ' / '1-1. '만 지우고 그대로 비교한다.

공백 차이도 다른 글자로 본다: "객실·부대시설" ≠ "객실 · 부대시설", "예약현황" ≠ "예약 현황".
기호 글꼴(PUA) 문자는 가운뎃점으로 바꿔 비교하고(ncs_core.make_normalizer), 그 사실은 비고에 적는다.
"""

from ncs_core import make_normalizer

HEAD_NUMBER = r"^\d+(-\d+)?\.\s*"
norm, same = make_normalizer(HEAD_NUMBER)


def split_items(text: str) -> list[str]:
    """쉼표로 나눈 항목 목록. 괄호 안의 쉼표(예: '체크 인(Check In, 투숙)')는 나누지 않는다."""
    items, depth, current = [], 0, []
    for ch in text:
        if ch in "([{":
            depth += 1
        elif ch in ")]}":
            depth = max(depth - 1, 0)
        if ch in ",，、" and depth == 0:
            items.append("".join(current))
            current = []
        else:
            current.append(ch)
    items.append("".join(current))
    return [item.strip() for item in items if item.strip()]
''')


In [ ]:
# 모듈: ncs_toc/__init__.py
write_module('ncs_toc/__init__.py', r'''"""F02 차례 ①·③·④·⑤·⑥ 검수. 읽기·판정·저장은 ncs_core를 쓴다."""

from ncs_core import ReviewResult, SavedReport

from .config import CheckConfig
from .pipeline import evaluate_review, prepare_review, run_review
from .registry import DEFAULT_RULES
from .reporting import save_results

__all__ = [
    "CheckConfig",
    "DEFAULT_RULES",
    "ReviewResult",
    "SavedReport",
    "prepare_review",
    "evaluate_review",
    "run_review",
    "save_results",
]
__version__ = "2.2.0"
''')


In [ ]:
# 모듈: ncs_toc/__main__.py
write_module('ncs_toc/__main__.py', r'''"""python -m ncs_toc 원고.hwpx [개발계획서.hwpx] [--module-code LM...] [--out 결과]"""

import argparse
import json

from . import CheckConfig, run_review, save_results


def main() -> None:
    parser = argparse.ArgumentParser(description="F02 차례 ①·③·④·⑤·⑥ 검수")
    parser.add_argument("paths", nargs="+", help="원고와 (선택) 개발계획서. 순서 무관, 내용으로 역할 판별")
    parser.add_argument("--manuscript", help="자동 역할 판별 대신 원고로 지정할 입력 경로")
    parser.add_argument("--plan", help="자동 역할 판별 대신 개발계획서로 지정할 입력 경로")
    parser.add_argument("--module-code", default="", help="비우면 원고 파일명에서 추출")
    parser.add_argument("--front-section", type=int, default=0)
    parser.add_argument("--body-section", type=int, default=1)
    parser.add_argument("--out", default="결과")
    args = parser.parse_args()
    roles = {}
    if args.manuscript:
        roles[args.manuscript] = "manuscript"
    if args.plan:
        roles[args.plan] = "plan"
    try:
        config = CheckConfig(front_section=args.front_section, body_section=args.body_section)
        result = run_review(args.paths, roles=roles, module_code=args.module_code, config=config)
        saved = save_results(result, args.out)
    except (OSError, ValueError) as error:
        parser.exit(2, f"검수 실행 실패: {error}\n")
    print(json.dumps({"요약": saved.summary, "진단": result.diagnostics}, ensure_ascii=False, indent=2))
    print(f"지적사항 {len(result.findings)}건 · {saved.zip_path}")


if __name__ == "__main__":
    main()
''')


In [ ]:
# 모듈: ncs_toc/compare.py
write_module('ncs_toc/compare.py', r'''"""목록 비교 도우미. 규칙 모듈이 함께 쓴다. 비교 정책은 v1.1을 유지한다."""

import difflib
from collections.abc import Iterable, Sequence

from ncs_core import O, PUA, X, visible

from .text import norm


def diff_list(label: str, expected: Sequence[str], actual: Sequence[str]) -> tuple[str, list[str]]:
    """difflib로 누락/추가/변경/순서를 구분한다 → (체크값, 비고 목록).

    중간에 하나가 빠져도 뒤가 연쇄로 '다름'이 되지 않는다.
    """
    exp, act = [norm(x) for x in expected], [norm(x) for x in actual]
    notes: list[str] = []
    if len(expected) != len(actual):
        notes.append(f"{label} 개수: 이 자리 {len(actual)} / 기준 {len(expected)}")
    if exp != act and sorted(exp) == sorted(act):
        notes.append(f"{label} 순서 다름: 이 자리 {[visible(x) for x in actual]} / 기준 {list(expected)}")
        return X, notes
    matcher = difflib.SequenceMatcher(None, exp, act, autojunk=False)
    for op, i1, i2, j1, j2 in matcher.get_opcodes():
        if op == "equal":
            continue
        if op == "replace" and i2 - i1 == j2 - j1:
            for k in range(i2 - i1):
                notes.append(
                    f'{label} {i1 + k + 1}번째 다름: "{visible(actual[j1 + k])}" / 기준 "{expected[i1 + k]}"'
                )
        else:
            for k in range(i1, i2):
                notes.append(f'{label} {k + 1}번째 누락: "{expected[k]}"')
            for k in range(j1, j2):
                notes.append(f'{label} 이 자리 {k + 1}번째가 기준에 없음: "{visible(actual[k])}"')
    return (O if not notes else X), notes


def pua_note(texts: Iterable[str], already: Iterable[str] = ()) -> tuple[str, list[str]]:
    """already: 차례(①)에서 이미 보고된 문자열(PUA→· 치환형). 같은 원인이면 X 대신 비고만."""
    already = set(already)
    bad = [t for t in texts if PUA.search(t)]
    new = [t for t in bad if PUA.sub("·", t) not in already]
    notes = [f"기호 글꼴 문자: {visible(t)}" for t in new]
    if bad and not new:
        notes.append("기호 글꼴 문자가 있으나 차례와 같은 원인 (①에서 보고됨)")
    return (X if new else O), notes
''')


In [ ]:
# 모듈: ncs_toc/config.py
write_module('ncs_toc/config.py', r'''"""차례 검수의 조정 가능한 양식 가정. 정규식 자체는 patterns.py에서 관리한다."""
from dataclasses import dataclass


@dataclass(frozen=True)
class CheckConfig:
    front_section: int = 0
    body_section: int = 1
    body_heading_max_length: int = 60
    eval_candidate_max_length: int = 30
    tail_min_paragraphs: int = 3
    reference_min_lines: int = 3
    ambiguous_tail_min_paragraphs: int = 10

    def __post_init__(self):
        if self.front_section < 0 or self.body_section < 0:
            raise ValueError("섹션 번호는 0 이상이어야 합니다")
        if self.front_section == self.body_section:
            raise ValueError("현재 파서는 차례·개요와 본문이 별도 섹션인 양식을 지원합니다")
        for name in ("body_heading_max_length", "eval_candidate_max_length",
                     "tail_min_paragraphs", "reference_min_lines",
                     "ambiguous_tail_min_paragraphs"):
            if getattr(self, name) < 1:
                raise ValueError(f"{name}은 1 이상이어야 합니다")


DEFAULT_CONFIG = CheckConfig()
''')


In [ ]:
# 모듈: ncs_toc/evaluation.py
write_module('ncs_toc/evaluation.py', r'''"""본문 표 가운데 평가 준거·평가 방법 표를 찾아 '학습 내용' 열 값을 꺼낸다."""

from ncs_core import Table

HEADER_KEYS = ("학습내용", "학습목표", "평가항목")


def evaluation_tables(tables: list[Table]) -> list[tuple[int, str, list[str]]]:
    """[(표가 있는 문단 인덱스, '평가 준거'|'평가 방법', ['학습 내용' 열 값 …])].

    '학습 내용' 머리글이 있고 '학습 목표'(평가 준거) 또는 '평가 항목'(평가 방법) 머리글이
    함께 있는 표만 평가 표로 본다.
    """
    out = []
    for table in tables:
        headers: dict[str, tuple[int, int]] = {}
        for (row, col), value in sorted(table["cells"].items()):
            key = next((k for k in HEADER_KEYS if value.replace(" ", "").startswith(k)), None)
            if key and key not in headers:
                headers[key] = (row, col)
        if "학습내용" not in headers or not ({"학습목표", "평가항목"} & headers.keys()):
            continue
        kind = "평가 준거" if "학습목표" in headers else "평가 방법"
        header_row, content_col = headers["학습내용"]
        values = [
            v for (r, c), v in sorted(table["cells"].items()) if c == content_col and r > header_row and v
        ]
        out.append((table["paragraph"], kind, values))
    return out
''')


In [ ]:
# 모듈: ncs_toc/models.py
write_module('ncs_toc/models.py', r'''"""파서와 규칙 사이의 경계. 결과 모델(ReviewResult 등)은 ncs_core.models를 쓴다."""

from dataclasses import dataclass
from typing import Any

from ncs_core import Document

from .config import CheckConfig


@dataclass(frozen=True)
class ReviewContext:
    """읽은 문서와 추출 결과. 규칙은 이 객체 하나를 받는다."""

    manuscript: Document
    plan: Document | None
    module_code: str
    config: CheckConfig
    toc: dict[str, Any]
    overview: dict[str, Any] | None
    plan_toc: dict[str, Any] | None
    body: dict[str, Any]
    presence: dict[str, Any]
    eval_tables: list[tuple[int, str, list[str]]]
''')


In [ ]:
# 모듈: ncs_toc/parsers.py
write_module('ncs_toc/parsers.py', r'''"""문단에서 차례, 개요, 본문 구조, 개발계획서 명칭을 추출한다.

sections는 [차례·개요 문단, 본문 문단]의 두 섹션 목록이다(pipeline이 실제 섹션을 연결한다).
"""

import re

from .config import DEFAULT_CONFIG, CheckConfig
from .patterns import (
    BODY_SUB,
    EVAL_ITEM_END,
    EVAL_LABELS,
    HEAD_EVAL,
    HEAD_TEACH,
    NAV_HEAD,
    OV_LESSON,
    OV_SUB,
    TOC_LESSON,
    TOC_LINE,
    TOC_SUB,
    TOC_TAIL,
)

TAIL_NAMES = {"참고자료": "참고 자료", "활용서식": "활용 서식", "부록": "부록"}


def parse_toc(sections) -> dict | None:
    """차례 블록 → {'학습': [{'no','text'}], '내용': [{'no','sub','text','page'}],
    '줄': [{'lesson','kind','page','after'}], '꼬리': {참고 자료: 쪽, ...}, 'span': (s, e)}"""
    lines = [l for l in sections[0] if l] if sections else []
    start = next((i for i, l in enumerate(lines) if re.fullmatch(r"차\s*례", l)), None)
    if start is None:
        return None
    end = next(
        (
            i
            for i in range(start + 1, len(lines))
            if re.search(r"학습모듈의\s*개요$", lines[i]) and not re.search(r"\d+$", lines[i])
        ),
        len(lines),
    )
    toc = {"학습": [], "내용": [], "줄": [], "꼬리": {}, "span": (start, end)}
    for line in lines[start + 1 : end]:
        if (m := TOC_LESSON.match(line)) and int(m.group(1)) == len(toc["학습"]) + 1:
            toc["학습"].append({"no": int(m.group(1)), "text": m.group(2).strip()})
        elif m := TOC_SUB.match(line):
            toc["내용"].append(
                {"no": int(m.group(1)), "sub": int(m.group(2)), "text": m.group(3).strip(), "page": m.group(4)}
            )
        elif m := TOC_LINE.match(line):
            kind = "교수·학습 방법" if "교수" in m.group(1) else "평가"
            lesson_no = len(toc["학습"])
            # after: 이 줄까지 읽은 해당 학습의 내용(N-M.) 줄 수. ④가 '블록 끝에 있는가'를 판단할 때 쓴다.
            after = sum(1 for s in toc["내용"] if s["no"] == lesson_no)
            toc["줄"].append({"lesson": lesson_no, "kind": kind, "page": m.group(2), "after": after})
        elif m := TOC_TAIL.match(line):
            toc["꼬리"][TAIL_NAMES[re.sub(r"\s+", "", m.group(1))]] = m.group(2)
    return toc


def parse_overview(sections) -> dict | None:
    """학습모듈의 개요 > 내용체계 → {'학습': [{'no','text'}], '내용': [{'no','sub','text'}]}"""
    lines = [l for l in sections[0] if l] if sections else []
    start = next((i for i, l in enumerate(lines) if re.fullmatch(r"학습모듈의\s*내용\s*체계", l)), None)
    if start is None:
        return None
    end = next((i for i in range(start + 1, len(lines)) if re.fullmatch(r"핵심\s*용어", lines[i])), len(lines))
    overview = {"학습": [], "내용": []}
    for line in lines[start + 1 : end]:
        if m := OV_SUB.match(line):
            overview["내용"].append({"no": int(m.group(1)), "sub": int(m.group(2)), "text": m.group(3).strip()})
        elif (m := OV_LESSON.match(line)) and int(m.group(1)) == len(overview["학습"]) + 1:
            overview["학습"].append({"no": int(m.group(1)), "text": m.group(2).strip()})
    return overview


def _navigator_runs(lines, n_lessons: int) -> list[list[dict]]:
    """내비게이터: '학습 1' 다음 줄이 이름, 이어서 '학습 2' …가 한 묶음(run)."""
    runs, k = [], 0
    while k < len(lines):
        head = NAV_HEAD.match(lines[k][1])
        if head and int(head.group(1)) == 1:
            run, no, j = [], 1, k
            while j + 1 < len(lines):
                nav = NAV_HEAD.match(lines[j][1])
                if not (nav and int(nav.group(1)) == no):
                    break
                run.append({"no": no, "text": lines[j + 1][1], "k": j})
                no += 1
                j += 2
            # 개수 차이는 ③ 내비게이터 항목의 비고로 보고한다.
            if len(run) >= (1 if n_lessons == 1 else 2):
                runs.append(run)
                k = j
                continue
        k += 1
    return runs


def _eval_candidates(lines, start: int, end: int, config: CheckConfig) -> list[str]:
    """평가 블록 안의 짧은 줄 = '학습 내용' 후보 (표를 못 읽을 때의 휴리스틱)."""
    out = []
    for _, line in lines[start:end]:
        if (
            len(line) <= config.eval_candidate_max_length
            and not re.match(r"^[-•·※\d(\[]", line)
            and line not in EVAL_LABELS
            and not NAV_HEAD.match(line)
            and not HEAD_EVAL.match(line)
            and not re.search(r"[.?!]$", line)
            and not EVAL_ITEM_END.search(line)
        ):
            out.append(line)
    return out


def parse_body(sections, n_lessons: int, cutoff: int | None = None, config: CheckConfig = DEFAULT_CONFIG) -> dict:
    """본문 → 내비게이터 묶음 / 학습 구간 / 필요지식 N-M. 제목 / 학습별 교수·평가 제목 위치와 평가 후보.

    cutoff: 참고 자료 시작 인덱스(raw) — 마지막 학습 구간을 여기서 끊는다.
    """
    raw = sections[1] if len(sections) > 1 else []
    lines = [(i, l) for i, l in enumerate(raw) if l]  # (원래 인덱스, 텍스트)
    nav_runs = _navigator_runs(lines, n_lessons)
    starts = [run[0]["k"] for run in nav_runs]
    last = next((k for k, (i, _) in enumerate(lines) if cutoff is not None and i >= cutoff), len(lines))
    spans = [(i + 1, starts[i], starts[i + 1] if i + 1 < len(starts) else last) for i in range(len(starts))]

    subs = []  # 필요지식 N-M. 제목 (같은 줄 또는 다음 줄)
    for k, (_, line) in enumerate(lines):
        if (m := BODY_SUB.match(line)) and len(line) < config.body_heading_max_length:
            text = m.group(3).strip() or (lines[k + 1][1] if k + 1 < len(lines) else "")
            if not re.search(r"\s\d+$", line):  # 차례 줄(끝 쪽수) 제외
                subs.append({"no": int(m.group(1)), "sub": int(m.group(2)), "text": text, "k": k})

    lessons = []
    for no, a, b in spans:
        segment = lines[a:b]
        teach = [a + j for j, (_, l) in enumerate(segment) if HEAD_TEACH.match(l)]
        evals = [a + j for j, (_, l) in enumerate(segment) if HEAD_EVAL.match(l)]
        candidates = _eval_candidates(lines, evals[0], b, config) if evals else []
        own_subs = [s for s in subs if a <= s["k"] < b and s["no"] == no]
        raw_end = lines[b][0] if b < len(lines) else (lines[-1][0] + 1 if lines else 0)
        lessons.append(
            {
                "no": no,
                "span": (a, b),
                "raw_span": (lines[a][0], raw_end),
                "teach": teach,
                "eval": evals,
                "eval_cands": candidates,
                "subs": own_subs,
            }
        )
    return {"lines": lines, "nav_runs": nav_runs, "spans": spans, "subs": subs, "lessons": lessons}


def parse_plan_toc(sections, module_code: str) -> dict | None:
    """개발계획서 → {'학습명': [...], '요소명': [...], '내용명': [{'no','sub','text'}]} (블록을 못 찾으면 None)"""
    lines = [l for section in sections for l in section]
    code = module_code.replace("LM", "")
    start = next(
        (i for i, l in enumerate(lines) if l == code and i > 0 and lines[i - 1].startswith("능력단위 코드번호")),
        None,
    )
    if start is None:
        return None
    end = next((i for i in range(start, len(lines)) if lines[i].startswith("NCS (수정 금지)")), len(lines))
    block = lines[start:end]
    plan = {"학습명": [], "요소명": [], "내용명": []}
    for i, line in enumerate(block):
        if re.fullmatch(rf"{re.escape(code)}\.\d+", line) and i + 1 < len(block):
            plan["요소명"].append(block[i + 1])
        if m := OV_SUB.match(line):
            plan["내용명"].append({"no": int(m.group(1)), "sub": int(m.group(2)), "text": m.group(3).strip()})
        elif re.fullmatch(r"\d+\.\s+\S.*", line):
            plan["학습명"].append(line)
    return plan
''')


In [ ]:
# 모듈: ncs_toc/patterns.py
write_module('ncs_toc/patterns.py', r'''"""양식별 제목·내용 인식 패턴. 판정 정책은 rules/에 둔다."""
import re

TOC_LESSON = re.compile(r'^학습\s*(\d+)\.\s*(.+?)(?:\s+\d+)?$')                 # 학습 1. 명 [쪽]
TOC_SUB    = re.compile(r'^(\d+)-(\d+)\.\s*(.+?)(?:\s*(\d+))?$')                 # 1-1. 명 [쪽]  ('• 평가52' 처럼 붙은 쪽수도)
TOC_LINE   = re.compile(r'^[•·]\s*(교수\s*\W{0,3}\s*학습\s*방법|평가)\s*(\d+)?$')
TOC_TAIL   = re.compile(r'^(참고\s*자료|활용\s*서식|부록)\s*(\d+)?$')
OV_LESSON  = re.compile(r'^(\d+)\.\s*(\S.*)$')                                   # 개요 내용체계: 1. 명
OV_SUB     = re.compile(r'^(\d+)-(\d+)\.\s*(\S.*)$')
NAV_HEAD   = re.compile(r'^학습\s*(\d+)$')
BODY_SUB   = re.compile(r'^(\d+)-(\d+)\.\s*(.*)$')
HEAD_TEACH = re.compile(r'^교수\s*\W{0,3}\s*학습\s*방법$')
HEAD_EVAL  = re.compile(r'^평\s*가$')
REF_LINE   = re.compile(r'^[•·]?\s*\S.{0,80}?\(\d{4}[a-z]?\)\s*[.,]')          # • 저자(연도).
EVAL_LABELS = {'평가 준거', '학습 내용', '학습 목표', '성취수준', '상', '중', '하', '평가 방법', '평가 항목', '피드백', '학습내용'}
EVAL_ITEM_END = re.compile(r'(여부|능력|수준|정도|가능)\s*$')     # 평가 항목 줄 종결어 (학습 내용명 후보에서 제외)
''')


In [ ]:
# 모듈: ncs_toc/pipeline.py
write_module('ncs_toc/pipeline.py', r'''"""입력 선택 → 추출 → 규칙 실행. 저장과 화면 출력은 호출자가 결정한다.

prepare_review: 문서를 읽고 ReviewContext를 만든다.
evaluate_review: 규칙을 실행해 ReviewResult를 만든다.
run_review: 둘을 이어 붙인 편의 함수.
"""

import re
from collections.abc import Sequence
from typing import Any

from ncs_core import Document, ReviewResult, Rule, resolve_module_code, run_rules, select_documents
from ncs_core.text import flatten

from .config import DEFAULT_CONFIG, CheckConfig
from .evaluation import evaluation_tables
from .models import ReviewContext
from .parsers import parse_body, parse_overview, parse_plan_toc, parse_toc
from .presence import section_presence
from .registry import ALL_ITEM_IDS, DEFAULT_RULES


def prepare_review(
    manuscript: Document,
    plan: Document | None = None,
    *,
    module_code: str = "",
    config: CheckConfig = DEFAULT_CONFIG,
) -> ReviewContext:
    code = resolve_module_code(manuscript.path, module_code)
    if max(config.front_section, config.body_section) >= len(manuscript.sections):
        raise ValueError("지정한 차례/본문 섹션이 없습니다. 문서 구조와 CheckConfig를 확인하세요")
    # 파서의 논리적 section0(차례·개요)/section1(본문)에 양식의 실제 섹션을 연결한다.
    sections = [manuscript.sections[config.front_section], manuscript.sections[config.body_section]]
    toc = parse_toc(sections)
    if not toc or not toc["학습"]:
        raise ValueError("차례 또는 차례 학습명을 읽지 못했습니다. 양식과 섹션 설정을 확인하세요")
    overview = parse_overview(sections)
    plan_toc = parse_plan_toc(plan.sections, code) if plan else None
    images = [(p, name) for s, p, name in manuscript.images if s == config.body_section]
    presence = section_presence(sections, images, config)
    body = parse_body(sections, len(toc["학습"]), cutoff=presence["참고 자료"]["k"], config=config)
    eval_tables = evaluation_tables(manuscript.section_tables(config.body_section))
    return ReviewContext(manuscript, plan, code, config, toc, overview, plan_toc, body, presence, eval_tables)


def _actual_sections(value: Any, config: CheckConfig) -> Any:
    """결과의 'section0/section1' 표기를 실제 입력 섹션 번호로 바꾼다."""
    if isinstance(value, str):
        return re.sub(
            r"section([01])\b",
            lambda m: "section" + str(config.front_section if m[1] == "0" else config.body_section),
            value,
        )
    if isinstance(value, dict):
        return {k: _actual_sections(v, config) for k, v in value.items()}
    if isinstance(value, list):
        return [_actual_sections(v, config) for v in value]
    return value


def _diagnostics(context: ReviewContext, rules: Sequence[Rule]) -> dict:
    manuscript, plan, config = context.manuscript, context.plan, context.config
    warnings = []
    if not context.plan_toc:
        warnings.append("개발계획서가 없거나 해당 모듈 블록을 읽지 못했습니다")
    elif not context.plan_toc["학습명"] or not context.plan_toc["내용명"]:
        warnings.append("개발계획서 명칭을 일부만 읽었습니다. 항목별 비교 기준을 확인하세요")
    if not context.body["lessons"]:
        warnings.append("본문 학습 구간을 읽지 못했습니다. 평가 대조와 ④ 교수·학습 방법/평가 위치는 판단불가입니다")
    if not context.eval_tables:
        warnings.append("평가 표 셀 정보가 없어 문단 휴리스틱으로 비교합니다")
    if len(manuscript.sections) > 2:
        warnings.append("선택한 차례/개요 및 본문 섹션만 검사합니다. 다른 섹션에 이어진 본문은 검사하지 않습니다")
    implemented = {rule.item_prefix for rule in rules}
    return {
        "module_code": context.module_code,
        "inputs": {"manuscript": str(manuscript.path), "plan": str(plan.path) if plan else None},
        "format": manuscript.format,
        "sections": {"front": config.front_section, "body": config.body_section},
        "counts": {
            "lessons": len(context.toc["학습"]),
            "contents": len(context.toc["내용"]),
            "navigator_runs": len(context.body["nav_runs"]),
            "evaluation_tables": len(context.eval_tables),
        },
        "implemented": sorted(implemented),
        "not_implemented": [item for item in ALL_ITEM_IDS if item not in implemented],
        "warnings": warnings,
    }


def evaluate_review(context: ReviewContext, *, rules: Sequence[Rule] = DEFAULT_RULES) -> ReviewResult:
    """한 실행의 결과만 돌려준다. 같은 context를 다시 실행해도 이전 결과를 누적하지 않는다."""
    config = context.config
    checklist = run_rules(context, rules)
    for row in checklist:
        row["region_path"] = _actual_sections(row["region_path"], config)
        row["비고"] = _actual_sections(row["비고"], config)
        if row["item_id"] == "F02-차례-⑤":
            row["원고"] = _actual_sections(row["원고"], config)
    diagnostics = _diagnostics(context, rules)
    return ReviewResult(context.module_code, checklist, flatten(checklist), diagnostics, context)


def run_review(
    paths,
    *,
    roles: dict | None = None,
    module_code: str = "",
    config: CheckConfig = DEFAULT_CONFIG,
    rules: Sequence[Rule] = DEFAULT_RULES,
) -> ReviewResult:
    """paths: 원고 1개 + 선택적 개발계획서 1개. roles={경로: 'manuscript'|'plan'}로 역할을 지정할 수 있다."""
    documents = select_documents(paths, roles, required=("manuscript",))
    manuscript, plan = documents["manuscript"], documents.get("plan")
    context = prepare_review(manuscript, plan, module_code=module_code, config=config)
    return evaluate_review(context, rules=rules)
''')


In [ ]:
# 모듈: ncs_toc/presence.py
write_module('ncs_toc/presence.py', r'''"""참고 자료·활용 서식·부록이 본문에 있는지의 근거를 찾는다."""

import re

from .config import DEFAULT_CONFIG, CheckConfig
from .patterns import REF_LINE

TAIL_PATTERNS = {"참고 자료": r"참\s*고\s*자\s*료", "활용 서식": r"활\s*용\s*서\s*식", "부록": r"부\s*록"}


def section_presence(sections, images, config: CheckConfig = DEFAULT_CONFIG) -> dict:
    """{이름: {'있음': bool|None, '근거': str, 'k': 위치}}.

    images: 본문 섹션의 (문단 인덱스, 원본 그림 이름). None은 '배너만 있고 내용은 없음'으로 사람 확인.
    """
    raw = sections[1] if len(sections) > 1 else []
    lines = [(i, l) for i, l in enumerate(raw) if l]
    out = {}
    for name, pattern in TAIL_PATTERNS.items():
        found = {"있음": False, "근거": "제목 텍스트·배너 이미지·내용 패턴 어느 것도 없음", "k": None}
        hit = next(((i, l) for i, l in lines if re.fullmatch(pattern, l)), None)
        if hit:  # 1) 제목 텍스트
            found = {"있음": True, "근거": f'제목 텍스트 "{hit[1]}" (section1 p{hit[0]})', "k": hit[0]}
        else:  # 2) 배너 이미지
            compact = pattern.replace(r"\s*", "")
            banner = next(((p, n) for p, n in images if re.search(compact, n.replace(" ", ""))), None)
            if banner:
                found = {"있음": True, "근거": f'배너 이미지 "{banner[1]}" (section1 p{banner[0]})', "k": banner[0]}
        out[name] = found

    # 제목·배너 뒤에 내용이 없으면(개발이력 전까지 문단이 적으면) '배너만' → △
    for name in ("활용 서식", "부록"):
        found = out[name]
        if not found["있음"]:
            continue
        others = [TAIL_PATTERNS[o] for o in TAIL_PATTERNS if o != name]
        stop = next(
            (
                i
                for i, l in lines
                if i > found["k"] and (re.search(r"개발\s*이력", l) or any(re.fullmatch(p, l) for p in others))
            ),
            len(raw),
        )
        body_n = len([l for i, l in lines if found["k"] < i < stop and not re.match(r"^※", l)])
        if body_n < config.tail_min_paragraphs:
            out[name] = {
                "있음": None,
                "근거": f'{found["근거"]} 뒤에 내용이 없음(문단 {body_n}개) — 양식 배너만 남은 것인지 확인',
                "k": found["k"],
            }

    # 3) 내용 패턴 — 참고 자료: 참고문헌 꼴 줄이 연속으로 충분히 있으면 존재로 본다
    if not out["참고 자료"]["있음"]:
        run, best = [], []
        for i, l in lines:
            if REF_LINE.match(l):
                run.append(i)
            else:
                if len(run) > len(best):
                    best = run
                run = []
        if len(run) > len(best):
            best = run
        if len(best) >= config.reference_min_lines:
            out["참고 자료"] = {
                "있음": True,
                "근거": f"참고문헌 꼴 줄 {len(best)}개 연속 (section1 p{best[0]}~p{best[-1]}) — 제목은 텍스트가 아님(이미지일 가능성)",
                "k": best[0],
                "end": best[-1],
            }

    # 활용 서식: 참고 자료 뒤 ~ 개발이력 앞에 내용이 있으면 '있을 가능성'
    if not out["활용 서식"]["있음"] and out["참고 자료"]["있음"]:
        start = out["참고 자료"].get("end", out["참고 자료"]["k"])
        stop = next((i for i, l in lines if i > start and re.search(r"개발\s*이력", l)), len(raw))
        middle = [l for i, l in lines if start < i < stop and not REF_LINE.match(l) and not re.match(r"^※", l)]
        if len(middle) >= config.ambiguous_tail_min_paragraphs:
            out["활용 서식"] = {
                "있음": None,
                "근거": (
                    f'제목은 없지만 참고 자료 뒤~개발이력 앞에 문단 {len(middle)}개 있음 (예: "{middle[0][:30]}") '
                    "— 활용 서식 또는 부록 내용으로 보임(제목이 이미지라 둘을 구분 못 함), 확인 필요"
                ),
                "k": start + 1,
            }
    return out
''')


In [ ]:
# 모듈: ncs_toc/registry.py
write_module('ncs_toc/registry.py', r'''"""규칙 추가 지점. 규칙은 ReviewContext 하나를 받아 기존 형식의 행 목록을 돌려준다."""

from ncs_core import Rule

from .rules.consistency import check_toc3
from .rules.names import check_toc1
from .rules.placement import check_toc4
from .rules.tail import check_toc5, check_toc6


def _check_toc4(c):
    images = [(p, name) for s, p, name in c.manuscript.images if s == c.config.body_section]
    return check_toc4(c.toc, c.body, images, lines_to_raw=lambda k: c.body["lines"][k][0])


DEFAULT_RULES = (
    Rule("names", "F02-차례-①", lambda c: check_toc1(c.plan_toc, c.toc)),
    Rule("consistency", "F02-차례-③", lambda c: check_toc3(c.plan_toc, c.toc, c.overview, c.body, c.eval_tables)),
    Rule("placement", "F02-차례-④", _check_toc4),
    Rule("references", "F02-차례-⑤", lambda c: check_toc5(c.presence, c.toc)),
    Rule("appendix", "F02-차례-⑥", lambda c: check_toc6(c.presence, c.toc)),
)

# 차례 항목 ①~⑦ 중 규칙이 없는 것은 진단의 not_implemented로 보고한다.
ALL_ITEM_IDS = tuple("F02-차례-" + n for n in "①②③④⑤⑥⑦")
''')


In [ ]:
# 모듈: ncs_toc/reporting.py
write_module('ncs_toc/reporting.py', r'''"""차례 검수 결과 저장. 파일 이름 규칙은 ncs_core.reporting과 같다."""

from ncs_core import ReviewResult, SavedReport, save_report

ITEM_FILES = (
    ("F02-차례-①", "차례①_학습명"),
    ("F02-차례-③", "차례③_학습명내용명_5곳일치"),
    ("F02-차례-④", "차례④_교수평가_위치"),
    ("F02-차례-⑤", "차례⑤_참고자료"),
    ("F02-차례-⑥", "차례⑥_활용서식부록"),
)


def save_results(result: ReviewResult, out_dir="결과", *, per_run_dir: bool = True) -> SavedReport:
    return save_report(
        result, out_dir, name="차례", item_files=ITEM_FILES, other_tail="차례_기타", per_run_dir=per_run_dir
    )
''')


In [ ]:
# 모듈: ncs_toc/rules/__init__.py
write_module('ncs_toc/rules/__init__.py', r'''"""문서 읽기와 독립적인 차례 검수 규칙."""
''')


In [ ]:
# 모듈: ncs_toc/rules/consistency.py
write_module('ncs_toc/rules/consistency.py', r'''"""F02 차례 ③: 개발계획서를 기준으로 다섯 자리(차례·개요·내비게이터·필요지식·평가)의 명칭을 비교한다.

개발계획서를 못 읽으면 차례를 기준으로 비교하되 △로 둔다.
"""

from ncs_core import O, PASS, PUA, TRI, UNDECIDED, verdict, visible, X

from ..compare import diff_list, pua_note
from ..text import norm, same

ITEM_ID = "F02-차례-③"


def _row(place, region_path, *, manuscript, reference, checks, notes, suggestion):
    result = verdict(checks)
    return {
        "item_id": ITEM_ID,
        "region_path": region_path,
        "자리": place,
        "원고": manuscript,
        "기준": reference,
        "체크": checks,
        "판정": result,
        "비고": notes,
        "수정 제안": None if result == PASS else suggestion,
    }


def _unreadable(place, region_path, check_name, note):
    return {
        "item_id": ITEM_ID,
        "region_path": region_path,
        "자리": place,
        "원고": None,
        "기준": None,
        "체크": {check_name: TRI},
        "판정": UNDECIDED,
        "비고": {check_name: note},
        "수정 제안": None,
    }


def check_toc3(plan, toc, overview, body, eval_tables=None) -> list[dict]:
    rows = []
    use_plan_names = bool(plan and plan["학습명"])
    use_plan_subs = bool(plan and plan["내용명"])
    ref_names = [norm(x) for x in plan["학습명"]] if use_plan_names else [L["text"] for L in toc["학습"]]
    ref_subs = plan["내용명"] if use_plan_subs else toc["내용"]
    ref_sub_texts = [s["text"] for s in ref_subs]
    toc_texts = [L["text"] for L in toc["학습"]] + [s["text"] for s in toc["내용"]]
    toc_pua = {PUA.sub("·", t) for t in toc_texts if PUA.search(t)}
    reference = {
        "학습명": ref_names,
        "학습 내용명": ref_sub_texts,
        "출처": "개발계획서" if use_plan_names and use_plan_subs else "차례" if not use_plan_names and not use_plan_subs else "혼합",
        "학습명 출처": "개발계획서" if use_plan_names else "차례",
        "학습 내용명 출처": "개발계획서" if use_plan_subs else "차례",
    }
    base_note = {}
    if not use_plan_names:
        base_note["기준"] = "개발계획서 학습명을 읽지 못해 차례를 기준으로 비교 (△)"
    if not use_plan_subs:
        base_note["기준(내용명)"] = "개발계획서 학습 내용명을 읽지 못해 차례를 기준으로 비교 (△)"

    def item(place, act_names, act_subs, region_path, extra_checks=None, extra_notes=None):
        checks, notes = {}, dict(base_note)
        if act_names is not None:
            value, found = diff_list("학습명", ref_names, act_names)
            checks["학습명 일치"] = value
            if found:
                notes["학습명 일치"] = " / ".join(found)
            if not use_plan_names and value == O and place != "차례":
                checks["학습명 일치"] = TRI
                notes["학습명 일치"] = "차례와 일치 (개발계획서 미확인)"
        if act_subs is not None:
            value, found = diff_list("학습 내용명", ref_sub_texts, act_subs)
            checks["학습 내용명 일치"] = value
            if found:
                notes["학습 내용명 일치"] = " / ".join(found)
            if not use_plan_subs and value == O and place != "차례":
                checks["학습 내용명 일치"] = TRI
                notes["학습 내용명 일치"] = "차례와 일치 (개발계획서 미확인)"
        value, found = pua_note((act_names or []) + (act_subs or []), toc_pua)
        checks["기호 글꼴 문자 없음"] = value
        if found:
            notes["기호 글꼴 문자 없음"] = " / ".join(found)
        if extra_checks:
            checks.update(extra_checks)
        if extra_notes:
            notes.update(extra_notes)
        rows.append(
            _row(
                place,
                region_path,
                manuscript={
                    "학습명": [visible(x) for x in act_names] if act_names is not None else None,
                    "학습 내용명": [visible(x) for x in act_subs] if act_subs is not None else None,
                },
                reference=reference,
                checks=checks,
                notes=notes,
                suggestion="기준 값으로 통일",
            )
        )

    # 1 차례 — 개발계획서 기준일 때만 의미 있음(차례 기준이면 자기 자신이므로 PUA만)
    if use_plan_names or use_plan_subs:
        item(
            "차례",
            [L["text"] for L in toc["학습"]] if use_plan_names else None,
            [s["text"] for s in toc["내용"]] if use_plan_subs else None,
            "section0 > 차례",
        )
    # 2 개요 내용체계
    if overview:
        item(
            "학습모듈의 개요 > 내용체계",
            [L["text"] for L in overview["학습"]],
            [s["text"] for s in overview["내용"]],
            "section0 > 학습모듈의 개요 > 내용체계 표",
        )
    else:
        rows.append(_unreadable("내용체계", "section0 > 학습모듈의 개요", "표 찾기", "'학습모듈의 내용체계' 표를 찾지 못함"))
    # 3 내비게이터 — 첫 묶음을 비교하고, 나머지 묶음의 차이는 추가 체크로
    if body["nav_runs"]:
        extra_checks, extra_notes = {}, {}
        for r_i, run in enumerate(body["nav_runs"][1:], 2):
            value, found = diff_list("학습명", ref_names, [x["text"] for x in run])
            if value == X:
                key = f"내비게이터 {r_i}번째(학습 {r_i}) 일치"
                extra_checks[key] = X
                extra_notes[key] = " / ".join(found)
        if len(body["nav_runs"]) != len(toc["학습"]):
            extra_checks["내비게이터 반복 횟수"] = TRI
            extra_notes["내비게이터 반복 횟수"] = (
                f"내비게이터 묶음 {len(body['nav_runs'])}개 / 차례 학습 {len(toc['학습'])}개 — 학습 구간이 어긋났을 수 있음"
            )
        item(
            "내비게이터",
            [x["text"] for x in body["nav_runs"][0]],
            None,
            f"section1 > 내비게이터 (학습마다 반복 {len(body['nav_runs'])}회)",
            extra_checks,
            extra_notes,
        )
    else:
        rows.append(_unreadable("내비게이터", "section1 > 내비게이터", "내비게이터 찾기", "'학습 1'…'학습 N' 제목 묶음을 찾지 못함"))
    # 4 필요지식 N-M. 제목 — 학습 구간 안의 것만, (no, sub) 순서로
    body_subs = sorted({(s["no"], s["sub"]): s["text"] for L in body["lessons"] for s in L["subs"]}.items())
    item("필요지식(본문) 학습 내용명", None, [t for _, t in body_subs], "section1 > 본문 N-M. 제목")
    # 5 평가 — 학습마다. 1순위: 표의 '학습 내용' 열(확정). 2순위: 블록 안 짧은 줄 휴리스틱(△)
    rows.append(_evaluation_row(body, ref_subs, ref_sub_texts, eval_tables, use_plan_subs, toc_pua, base_note))
    return rows


def _evaluation_row(body, ref_subs, ref_sub_texts, eval_tables, use_plan_subs, toc_pua, base_note):
    checks, notes, seen = {}, {}, []
    by_table = bool(eval_tables)
    for lesson in body["lessons"]:
        expected = [s["text"] for s in ref_subs if s["no"] == lesson["no"]]
        key = f"학습 {lesson['no']} 평가"
        if by_table:
            start, end = lesson["raw_span"]
            tables = [(kind, values) for paragraph, kind, values in eval_tables if start <= paragraph < end]
            if not tables:
                checks[key] = X
                notes[key] = "학습 구간 안에 평가 준거·평가 방법 표('학습 내용' 열)를 찾지 못함"
                continue
            problems = []
            for kind, values in tables:
                seen += values
                value, found = diff_list(f"{kind} 학습 내용", expected, values)
                if value == X:
                    problems += found
            checks[key] = O if not problems else X
            if problems:
                notes[key] = " / ".join(problems)
            kinds = {kind for kind, _ in tables}
            missing = [k for k in ("평가 준거", "평가 방법") if k not in kinds]
            if missing:
                checks[key] = TRI if checks[key] == O else checks[key]
                notes[key] = (notes.get(key, "") + f" / {', '.join(missing)} 표('학습 내용' 열)를 찾지 못함").strip(" /")
        else:
            if not lesson["eval"]:
                checks[key] = X
                notes[key] = "'평 가' 제목을 찾지 못함"
                continue
            candidates = lesson["eval_cands"]
            seen += candidates
            missing = [e for e in expected if not any(same(e, c) for c in candidates)]
            strange = sorted({c for c in candidates if not any(same(e, c) for e in expected)})
            if missing:
                checks[key] = X
                notes[key] = "평가 표에 없는 학습 내용명: " + ", ".join(f'"{m}"' for m in missing)
            elif strange:
                checks[key] = TRI
                notes[key] = "기준에 없는 짧은 줄(학습 내용명일 수 있음): " + ", ".join(f'"{visible(x)}"' for x in strange[:5])
            else:
                checks[key] = TRI
                notes[key] = "표를 못 읽어 문단 휴리스틱으로 확인 (hwpx 권장)"
    if not body["lessons"]:
        checks["평가 구간 찾기"] = TRI
        notes["평가 구간 찾기"] = "학습 구간을 읽지 못해 평가 표 대조를 수행하지 못함"
    if not use_plan_subs:
        checks["개발계획서 기준 확인"] = TRI
        notes["개발계획서 기준 확인"] = "개발계획서 학습 내용명을 읽지 못해 차례를 기준으로 비교"
    value, found = pua_note(seen, toc_pua)
    checks["기호 글꼴 문자 없음"] = value
    all_notes = dict(base_note)
    all_notes.update(notes)
    if found:
        all_notes["기호 글꼴 문자 없음"] = " / ".join(found)
    return _row(
        "평가",
        "section1 > 평가 (평가 준거·평가 방법 표)",
        manuscript={
            "평가 표 학습 내용" if by_table else "학습 내용명 후보": sorted(set(seen)),
            "읽기": "표 셀" if by_table else "문단 휴리스틱",
        },
        reference={"학습 내용명": ref_sub_texts, "출처": "개발계획서" if use_plan_subs else "차례"},
        checks=checks,
        notes=all_notes,
        suggestion="평가 준거·평가 방법 표의 학습 내용명을 기준 값으로 통일",
    )
''')


In [ ]:
# 모듈: ncs_toc/rules/names.py
write_module('ncs_toc/rules/names.py', r'''"""F02 차례 ①: 학습명 — '~하기' 종결, 개발계획서 학습명 일치, 능력단위요소명 준용, 기호 글꼴 문자."""

import re

from ncs_core import FAIL, NA, O, PASS, PUA, TRI, verdict, visible, X

from ..text import norm, same

ITEM_ID = "F02-차례-①"


def check_toc1(plan: dict | None, toc: dict) -> list[dict]:
    rows = []
    n_plan = len(plan["학습명"]) if plan else None
    if plan and n_plan != len(toc["학습"]):
        rows.append(
            {
                "item_id": ITEM_ID,
                "region_path": "section0 > 차례",
                "원고": f"학습 {len(toc['학습'])}개",
                "개발계획서": f"학습 {n_plan}개",
                "체크": {"학습 개수 일치": X},
                "판정": FAIL,
                "비고": {"학습 개수 일치": "학습 누락 또는 추가"},
                "수정 제안": None,
            }
        )
    for index, lesson in enumerate(toc["학습"]):
        text = lesson["text"]
        exp_name = plan["학습명"][index] if plan and index < n_plan else None
        exp_elem = plan["요소명"][index] if plan and index < len(plan["요소명"]) else None
        ok_name = same(exp_name, text) if exp_name else None
        ok_elem = same(exp_elem, text) if exp_elem else None
        ws_only = ok_elem is False and re.sub(r"\s", "", norm(exp_elem)) == re.sub(r"\s", "", norm(text))
        if ws_only:
            ok_elem = True
        checks = {
            "'~하기' 종결": O if re.search(r"하기$", text) else X,
            "개발계획서 학습명 일치": NA if ok_name is None else (O if ok_name else X),
            "능력단위요소명 준용": NA if ok_elem is None else (O if ok_elem else TRI),
            "기호 글꼴 문자 없음": X if PUA.search(text) else O,
        }
        notes = {}
        if checks["'~하기' 종결"] == X:
            notes["'~하기' 종결"] = "'~하기'로 종결되지 않음"
        if ok_name is False:
            notes["개발계획서 학습명 일치"] = f'개발계획서 학습명과 다름: "{exp_name}"'
        if ok_elem is False:
            notes["능력단위요소명 준용"] = f'능력단위요소명 "{exp_elem}" 과 글자가 다름 — 요소를 합치거나 이름을 바꿨는지 확인'
        if ws_only:
            notes["능력단위요소명 준용(참고)"] = f'능력단위요소명 "{exp_elem}" 과 공백만 다름 — 통과 처리'
        if PUA.search(text):
            notes["기호 글꼴 문자 없음"] = "가운뎃점 등이 기호 글꼴(사용자 정의 영역) 문자 — ·(U+00B7)로 교체"
        if not plan:
            notes["개발계획서"] = "개발계획서를 읽지 못해 학습명·요소명 대조는 생략"
        result = verdict(checks)
        rows.append(
            {
                "item_id": ITEM_ID,
                "region_path": f"section0 > 차례 > 학습 {lesson['no']}",
                "원고": visible(text),
                "개발계획서": norm(exp_name) if exp_name else None,
                "능력단위요소명": exp_elem,
                "체크": checks,
                "판정": result,
                "비고": notes,
                "수정 제안": norm(exp_name) if (result != PASS and exp_name) else None,
            }
        )
    return rows
''')


In [ ]:
# 모듈: ncs_toc/rules/placement.py
write_module('ncs_toc/rules/placement.py', r'''"""F02 차례 ④: '교수·학습 방법'과 '평가'가 각 학습(대단원) 끝부분에 있는가. (필수 작성)

차례 면: 각 학습 블록에서 'N-M.' 내용 줄이 모두 나온 뒤 '• 교수·학습 방법', '• 평가' 줄이 있는가.
본문:   각 학습 구간에서 마지막 '필요지식 N-M.' 제목 뒤에 교수·학습 방법 제목, 그 뒤에 평가 제목이 오는가.
        제목이 글자가 아닌 배너 이미지(원본 그림 이름에 '교수'/'평가')면 그 위치를 제목으로 본다.
"""

import re

from ncs_core import O, PASS, TRI, X, verdict

ITEM_ID = "F02-차례-④"
KIND_TEACH, KIND_EVAL = "교수·학습 방법", "평가"
BANNER = {KIND_TEACH: re.compile(r"교수"), KIND_EVAL: re.compile(r"평가")}


def _toc_lines(toc: dict, no: int) -> dict[str, dict]:
    """해당 학습의 차례 줄 {'교수·학습 방법': 줄, '평가': 줄}. 같은 종류가 둘이면 첫 줄."""
    out: dict[str, dict] = {}
    for line in toc["줄"]:
        if line["lesson"] == no and line["kind"] not in out:
            out[line["kind"]] = line
    return out


def _toc_check(toc: dict, no: int) -> tuple[str, str | None]:
    """차례 줄 유무와 위치 → (체크값, 비고)."""
    lines = _toc_lines(toc, no)
    n_contents = sum(1 for s in toc["내용"] if s["no"] == no)
    missing = [kind for kind in (KIND_TEACH, KIND_EVAL) if kind not in lines]
    if missing:
        return X, f"차례 학습 {no} 블록에 {' / '.join(missing)} 줄이 없음"
    # 'after': 그 줄이 나올 때까지 읽은 이 학습의 내용 줄 수. 내용 줄 전체 수보다 적으면 중간에 끼어 있다.
    early = [kind for kind, line in lines.items() if line.get("after", n_contents) < n_contents]
    if early:
        return X, f"차례 학습 {no} 블록에서 {' / '.join(early)} 줄이 내용(N-M.) 줄 앞에 있음 — 블록 끝으로 이동"
    if lines[KIND_TEACH].get("after", 0) > lines[KIND_EVAL].get("after", 0):
        return X, f"차례 학습 {no} 블록에서 평가 줄이 교수·학습 방법 줄보다 앞에 있음"
    return O, None


def _banner_positions(images, raw_span: tuple[int, int], kind: str) -> list[int]:
    """학습 구간(raw 문단 범위) 안에서 kind 제목에 해당하는 배너 이미지의 raw 문단 인덱스."""
    a, b = raw_span
    pattern = BANNER[kind]
    return [p for p, name in images if a <= p < b and pattern.search(name.replace(" ", ""))]


def _heading(lesson: dict, images, kind: str) -> tuple[int | None, str]:
    """제목 위치(raw 문단 인덱스)와 근거. 글자 제목 우선, 없으면 배너 이미지."""
    lines_index = lesson["teach"] if kind == KIND_TEACH else lesson["eval"]
    if lines_index:
        return lines_index[0], "제목 텍스트"
    banners = _banner_positions(images, lesson["raw_span"], kind)
    if banners:
        return banners[0], "배너 이미지"
    return None, "없음"


def check_toc4(toc: dict, body: dict, images, *, lines_to_raw) -> list[dict]:
    """images: 본문 섹션의 (raw 문단 인덱스, 원본 그림 이름). lines_to_raw: lines 인덱스 → raw 문단 인덱스."""
    by_no = {lesson["no"]: lesson for lesson in body["lessons"]}
    rows = []
    for entry in toc["학습"]:
        no, name = entry["no"], entry["text"]
        label_toc = "④ 차례 줄(교수·학습 방법/평가) 위치"
        label_teach = "④ 본문 교수·학습 방법 제시"
        label_eval = "④ 본문 평가 제시"
        label_order = "④ 본문 끝부분 배치(필요지식 → 교수·학습 방법 → 평가)"
        notes: dict[str, str] = {}
        checks = {label_toc: O, label_teach: O, label_eval: O, label_order: O}
        evidence: dict = {"학습": f"학습 {no}. {name}"}

        toc_value, toc_note = _toc_check(toc, no)
        checks[label_toc] = toc_value
        if toc_note:
            notes[label_toc] = toc_note

        lesson = by_no.get(no)
        if lesson is None:
            checks[label_teach] = checks[label_eval] = checks[label_order] = TRI
            notes[label_order] = f"본문에서 학습 {no} 구간을 읽지 못함(내비게이터 기준) — 사람 확인"
            evidence["본문"] = None
        else:
            # lines 인덱스를 raw 문단 인덱스로 통일해 비교한다.
            teach_lines, teach_basis = _heading(lesson, images, KIND_TEACH)
            eval_lines, eval_basis = _heading(lesson, images, KIND_EVAL)
            teach_pos = lines_to_raw(teach_lines) if teach_basis == "제목 텍스트" else teach_lines
            eval_pos = lines_to_raw(eval_lines) if eval_basis == "제목 텍스트" else eval_lines
            last_sub = max((lines_to_raw(s["k"]) for s in lesson["subs"]), default=None)
            evidence["본문"] = {
                "구간": f"section1 p{lesson['raw_span'][0]}~p{lesson['raw_span'][1] - 1}",
                "마지막 필요지식 제목": f"p{last_sub}" if last_sub is not None else None,
                KIND_TEACH: f"p{teach_pos} ({teach_basis})" if teach_pos is not None else None,
                KIND_EVAL: f"p{eval_pos} ({eval_basis})" if eval_pos is not None else None,
            }
            if teach_pos is None:
                checks[label_teach] = X
                notes[label_teach] = f"학습 {no} 구간에 '교수·학습 방법' 제목(텍스트·배너)이 없음 (필수 작성)"
            if eval_pos is None:
                checks[label_eval] = X
                notes[label_eval] = f"학습 {no} 구간에 '평가' 제목(텍스트·배너)이 없음 (필수 작성)"
            if teach_pos is None or eval_pos is None:
                checks[label_order] = TRI if (teach_pos is None) != (eval_pos is None) else X
                notes.setdefault(label_order, "제목이 없어 배치를 판단할 수 없음")
            else:
                problems = []
                if eval_pos < teach_pos:
                    problems.append("평가가 교수·학습 방법보다 앞에 있음")
                if last_sub is not None and last_sub > min(teach_pos, eval_pos):
                    problems.append(f"필요지식 제목(p{last_sub})이 교수·학습 방법/평가 뒤에 있음 — 학습 끝부분이 아님")
                if not lesson["subs"]:
                    checks[label_order] = TRI
                    notes[label_order] = f"학습 {no} 구간에서 필요지식 N-M. 제목을 읽지 못해 '끝부분'을 확정할 수 없음"
                elif problems:
                    checks[label_order] = X
                    notes[label_order] = "; ".join(problems)

        result = verdict(checks)
        rows.append(
            {
                "item_id": ITEM_ID,
                "region_path": f"section1 > 학습 {no}",
                "원고": evidence,
                "체크": checks,
                "판정": result,
                "비고": notes,
                "수정 제안": None
                if result == PASS
                else "각 학습의 필요지식 뒤에 '교수·학습 방법' → '평가' 순으로 제시하고 차례에도 같은 순서로 줄을 둘 것",
            }
        )
    if not rows:
        rows.append(
            {
                "item_id": ITEM_ID,
                "region_path": "section0 > 차례",
                "원고": None,
                "체크": {"④ 교수·학습 방법/평가 위치": TRI},
                "판정": verdict({"x": TRI}),
                "비고": {"④ 교수·학습 방법/평가 위치": "차례에서 학습을 읽지 못해 판단할 수 없음"},
                "수정 제안": None,
            }
        )
    return rows
''')


In [ ]:
# 모듈: ncs_toc/rules/tail.py
write_module('ncs_toc/rules/tail.py', r'''"""F02 차례 ⑤·⑥: 참고 자료, 활용 서식·부록의 차례↔본문 일치."""

from ncs_core import O, PASS, TRI, verdict, X

ITEM_ID_5 = "F02-차례-⑤"
ITEM_ID_6 = "F02-차례-⑥"


def check_toc5(presence: dict, toc: dict) -> list[dict]:
    found = presence["참고 자료"]
    in_toc = "참고 자료" in toc["꼬리"]
    checks = {
        "⑤ 참고 자료 본문 존재": O if found["있음"] else X,
        "차례에 참고 자료 줄": O if in_toc else X,
    }
    notes = {"근거": found["근거"]}
    if not in_toc:
        notes["차례에 참고 자료 줄"] = '차례에 "참고 자료" 줄이 없음'
    if not found["있음"]:
        notes["⑤ 참고 자료 본문 존재"] = "본문에서 참고 자료를 찾지 못함 (필수 작성)"
    result = verdict(checks)
    return [
        {
            "item_id": ITEM_ID_5,
            "region_path": "section1 > 참고 자료",
            "원고": found["근거"],
            "차례": toc["꼬리"].get("참고 자료"),
            "체크": checks,
            "판정": result,
            "비고": notes,
            "수정 제안": None if result == PASS else "참고 자료를 작성하고 차례에 쪽수와 함께 제시",
        }
    ]


def check_toc6(presence: dict, toc: dict) -> list[dict]:
    rows = []
    for name in ("활용 서식", "부록"):
        found, in_toc = presence[name], name in toc["꼬리"]
        toc_page = toc["꼬리"][name] or "" if in_toc else ""
        if found["있음"] is None:
            value = TRI
            note = f'본문: {found["근거"]}' + (f" / 차례에는 {name} {toc_page}" if in_toc else " / 차례에는 없음")
        elif in_toc and found["있음"]:
            value, note = O, f'차례·본문 모두 있음 — {found["근거"]}'
        elif in_toc and not found["있음"]:
            value, note = X, f'차례에는 "{name} {toc_page}" 이 있는데 본문에 없음 — 미작성이면 차례에서 삭제'
        elif not in_toc and found["있음"]:
            value, note = X, f'본문에 {name} 이 있는데({found["근거"]}) 차례에 없음 — 차례에 추가'
        else:
            value, note = O, "차례·본문 모두 없음 (해당 없음)"
        label = f"⑥ {name} 차례↔본문 일치"
        checks = {label: value}
        if value == O:
            suggestion = None
        elif in_toc and found["있음"] is False:
            suggestion = "차례에서 삭제"
        elif found["있음"] and not in_toc:
            suggestion = "차례에 추가"
        else:
            suggestion = "본문 확인 후 차례와 맞출 것"
        rows.append(
            {
                "item_id": ITEM_ID_6,
                "region_path": f"section0 > 차례 > {name}",
                "원고": {"차례": toc["꼬리"].get(name) if in_toc else None, "본문": found["있음"]},
                "체크": checks,
                "판정": verdict(checks),
                "비고": {label: note} if value != O or "해당 없음" in note else {},
                "수정 제안": suggestion,
            }
        )
    return rows
''')


In [ ]:
# 모듈: ncs_toc/text.py
write_module('ncs_toc/text.py', r'''"""차례 검수의 문자 비교 규칙. 머리 번호 '학습 1. ' / '1. ' / '1-1. '을 지우고 비교한다."""

from ncs_core import make_normalizer

HEAD_NUMBER = r"^(학습\s*)?\d+(-\d+)?\.\s*"
norm, same = make_normalizer(HEAD_NUMBER)
''')


In [ ]:
# 재실행해도 이전 버전의 import 캐시를 사용하지 않습니다.
for name in list(sys.modules):
    if name.split(".")[0] in ("ncs_core", "ncs_toc", "ncs_overview", "ncs_knowledge"):
        del sys.modules[name]
if str(RUNTIME_DIR) not in sys.path:
    sys.path.insert(0, str(RUNTIME_DIR))
importlib.invalidate_caches()
from ncs_knowledge import CheckConfig, run_review, save_results
from ncs_core import select_documents

print("검수 엔진 준비 완료")


## 3. 입력과 설정
`upload`: 원고와 개발계획서를 한 번에 선택합니다(내용으로 역할 판별).
`drive`: Google Drive를 마운트하고 `PATHS`에 경로를 입력합니다. `local`: 로컬 Jupyter 경로를 입력합니다.
파일명에 모듈 코드가 없거나 둘 이상이면 `MODULE_CODE`를 지정하세요. 역할 판별이 안 되면 `ROLES`를 지정하세요.


In [ ]:
INPUT_METHOD = "upload"  # upload / drive / local
PATHS = []  # drive·local: 원고와 개발계획서 경로. upload: 비워 두면 다음 셀에서 선택
ROLES = {}  # 역할 자동 판별이 안 될 때: {"원고.hwpx": "manuscript", "개발계획서.hwpx": "plan"}
MODULE_CODE = ""  # 예: "LM1203020201_24v3". 비우면 원고 파일명에서 추출
CONFIG = CheckConfig(front_section=0, body_section=1, whitespace_is_difference=True)
OUTPUT_DIR = "결과"


In [ ]:
result = saved = None
if INPUT_METHOD == "upload":
    from google.colab import files
    upload_root = Path(tempfile.mkdtemp(prefix="ncs_inputs_"))
    print("원고와 개발계획서를 선택하세요 (여러 개 선택 가능).")
    uploaded = files.upload()
    PATHS = []
    for filename, content in uploaded.items():
        path = upload_root / Path(filename).name
        path.write_bytes(content)
        PATHS.append(str(path))
elif INPUT_METHOD == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
elif INPUT_METHOD != "local":
    raise ValueError("INPUT_METHOD는 upload / drive / local 중 하나여야 합니다")
if not PATHS:
    raise ValueError("입력 파일이 없습니다. PATHS를 지정하거나 파일을 업로드하세요")
documents = select_documents(PATHS, ROLES, required=('manuscript',))
for role, document in documents.items():
    print(role, "→", document.path.name, f"({document.format}, 섹션 {len(document.sections)}개)")


## 4. 검사와 추출 결과 확인
먼저 진단의 `warnings`와 `counts`를 확인하세요.


In [ ]:
result = run_review(PATHS, roles=ROLES, module_code=MODULE_CODE, config=CONFIG)
print(json.dumps(result.diagnostics, ensure_ascii=False, indent=2))
ctx = result.context
print("내비게이터(첫 묶음):", [x["text"] for x in ctx.body["nav_runs"][0]] if ctx.body["nav_runs"] else "없음")
print("학습 내용 구간 (N-M / 제목 / 놓인 학습 / 학습 목표 문장 수):")
for u in ctx.units:
    print(f"   {u['no']}-{u['sub']}. {u['text']}  학습 {u['lesson']}  목표 {len(u['goal'])}문장")
print("개발계획서:", f"학습 내용 {len(ctx.plan_goals['내용'])}개" if ctx.plan_goals else "없음/못 읽음")
print("캡션:", [c["label"] for c in ctx.labels["caps"][:12]], "…" if len(ctx.labels["caps"]) > 12 else "")


## 5. 저장과 결과 보기
결과는 실행별 하위 폴더에 저장됩니다.


In [ ]:
if result is None:
    raise ValueError("검사 셀을 먼저 정상 완료하세요")
saved = save_results(result, OUTPUT_DIR)
print(json.dumps(saved.summary, ensure_ascii=False, indent=2))
print(f"지적사항 {len(result.findings)}건")
for finding in result.findings:
    print(finding["item_id"], "|", finding["check"], "|", finding["reason"])
try:
    import pandas as pd
    from IPython.display import display
except ImportError:
    print(json.dumps(result.checklist, ensure_ascii=False, indent=2))
else:
    rows = [{"항목": r["item_id"], "구역": r["region_path"], "판정": r["판정"], "체크": r["체크"], "비고": r["비고"]}
            for r in result.checklist]
    display(pd.DataFrame(rows))
print("결과 ZIP:", saved.zip_path)


In [ ]:
if saved is None:
    raise ValueError("저장 셀을 먼저 정상 완료하세요")
try:
    from google.colab import files
except ImportError:
    print("ZIP 위치:", saved.zip_path)
else:
    files.download(str(saved.zip_path))


## 팀 공유 시 읽는 순서
1. `ncs_knowledge/pipeline.py`: 검사 전체 흐름. 어떤 파서를 어느 패키지에서 가져오는지.
2. `ncs_knowledge/models.py`: 파서와 규칙이 주고받는 데이터.
3. `ncs_knowledge/structure.py`(학습 내용 구간·hwpx 표 셀 문단), `plan.py`(계획서 학습 목표), `figtab.py`(그림·표 캡션·번호).
4. `ncs_knowledge/rules/`: navigator ①②③ · contents ④ · goals ⑤ · numbering ⑥ · figures ⑦ · sources ⑧. `registry.py`: 새 규칙을 연결할 곳.
5. `ncs_toc/parsers.py`·`presence.py`, `ncs_overview/parsers.py`: 함께 쓰는 파서. `ncs_core/`: 문서 읽기(hwp/hwpx), 정규화·판정, 입력 역할 판별, 모듈 코드, 저장.

- 개발계획서 없이 실행하면 ⑤는 `-`(대조 생략)이고 나머지는 같습니다.
- `필요지식_진단.json`의 `warnings`·`counts`로 내비게이터·학습 내용 구간·캡션이 제대로 잡혔는지 먼저 확인하세요.
- `*_전체.json`은 모든 결과, `*_지적사항.json`은 X·△만, `*_진단.json`은 입력·읽기 상태를 담습니다.
- 결과는 실행마다 `결과/<모듈코드>_xxxx/` 하위 폴더에 저장되어 이전 결과를 덮어쓰지 않습니다.

남은 것: ⑧ AI 활용 이미지 출처의 직능연 지정 양식 확인 → `rules/sources.py`의 `TEMPLATES`에 추가. ⑥ 번호 없는 소제목은 다음 번호에서 간접 지적됨.
페이지 좌표나 OCR을 사용하는 검사가 아니며, 이미지 속 글자를 읽지는 않습니다.
